## Introduction


Dans le cadre de notre projet de recommandation, nous (Noel Snelson, Kancel Johnathan, Ibrahim Assebbane et Zekri Rayane) exploitons le dataset MovieLens 20M (Kaggle). Il contient principalement :

ratings.csv : (userId, movieId, rating, timestamp) — ~20M interactions

movies.csv : (movieId, title, genres)

tags.csv : (userId, movieId, tag, timestamp)

genome-tags.csv / genome-scores.csv : tags normalisés + scores de pertinence

Notre objectif est de construire, pour chaque film, un texte explicable (titre + genres + tags, éventuellement enrichi via le genome) puis de générer des embeddings à l’aide de modèles de type Sentence-BERT. Nous utilisons ces représentations pour développer un système de recommandation content-based (similarité cosinus entre profil utilisateur et items), et le comparer à des baselines (popularité, MF/BPR via Cornac).

Compte tenu du volume, nous ne travaillons pas systématiquement sur l’intégralité des 20M ratings. Nous appliquons un filtrage (seuils configurables) :

utilisateurs avec un minimum d’interactions,

films avec un minimum de notations,

puis, si nécessaire, un sous-échantillonnage pour itérer rapidement lors des phases de test. Le run complet est utilisé ensuite pour stabiliser les résultats.

Enfin, nous structurons le notebook de manière progressive :

chargement et préparation des données + split (anti-fuite temporelle si possible),

baselines (popularité, collaboratif),

construction texte → embeddings → recommandation,

évaluation (Recall@K, NDCG@K, MAP@K, HR@K) + export des artefacts pour le rapport.

## Installation des dependances & les library (Colab) (kaggle)

In [13]:
!pip -q install kagglehub transformers sentence-transformers cornac
import importlib
missing = []
for pkg in ["kagglehub", "transformers", "sentence_transformers", "cornac", "torch"]:
    try:
        importlib.import_module(pkg)
    except Exception as exc:
        missing.append(f"{pkg}: {exc}")

if missing:
    print("Warning: missing packages after install:")
    for msg in missing:
        print(" -", msg)
else:
    print("Dependencies OK")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.6/29.6 MB 67.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 12.7 MB/s eta 0:00:00
Dependencies OK


Commentaire – Installation des dépendances + vérification des imports

Nous installons ici les bibliothèques nécessaires au notebook :

- kagglehub : télécharger/charger MovieLens depuis Kaggle.
- transformers : utiliser des modèles Hugging Face (tokenizers + modèles).
- sentence-transformers : générer des embeddings de textes (MiniLM, MPNet, etc.).
- cornac : entraîner des baselines collaboratives (MF, BPR) pour comparer nos approches.
- torch : dépendance centrale pour exécuter la majorité des modèles d’embeddings.

L’option `-q` (quiet) réduit les logs pour garder une sortie plus lisible.

Ensuite, nous vérifions explicitement que chaque package est bien importable. C’est une étape de robustesse : en environnement notebook (Colab/Kaggle), une installation peut échouer partiellement ou entrer en conflit.  
- Si tout est correct : “Dependencies OK”.
- Sinon : nous listons précisément les modules manquants et l’erreur associée, pour diagnostiquer rapidement.


In [14]:
import os
import json
import re
from datetime import datetime
import pandas as pd
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import train_test_split

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel


Commentaire – Imports principaux (I/O, traitement, ML et modèles)

Nous regroupons ici les imports nécessaires pour l’ensemble du pipeline de recommandation.

- os, json : gestion des chemins, lecture/écriture d’artefacts (ex. run_config.json, exports).
- re : nettoyage et normalisation de texte (ex. tags, titres, séparation de tokens).
- datetime : horodatage des runs et traçabilité des exports.

- pandas, numpy : manipulation efficace des tables (ratings/movies/tags), agrégations, filtrage, préparation des splits, calculs de métriques.

- kagglehub (+ KaggleDatasetAdapter) : récupération et chargement du dataset MovieLens depuis Kaggle directement dans le notebook.

- train_test_split (sklearn) : split train/test (utile pour un split aléatoire ou comme fallback si le split temporel n’est pas activé).

- torch : backend de calcul utilisé par les modèles (CPU/GPU) et dépendance de Transformers/Sentence-Transformers.

- SentenceTransformer : calcul des embeddings de films (approches content-based) via des modèles type MiniLM/MPNet/bge.

- AutoTokenizer, AutoModel (Transformers) : interface générique Hugging Face pour charger un tokenizer et un modèle (utile si nous testons une variante d’embeddings hors Sentence-Transformers, ou pour des modèles spécifiques).


-----

## Paramètres d’expérience (config du run) + reproductibilité

In [15]:
RANDOM_STATE = 42

MIN_USER_RATINGS = 15
MIN_ITEM_RATINGS = 50
MAX_INTERACTIONS = None  # safe default for Colab; increase for final runs

TEST_SIZE = 0.2
MIN_INTERACTIONS_PER_USER = 2
MIN_RATING_POSITIVE = 4.0

SPLIT_STRATEGY = "time"  # "time" or "random"

USER_PROFILE_MODE = "mean"  # "mean", "rating", "recency", "rating_recency"
RECENCY_LAMBDA = 0.01

USE_HYBRID = True
ALPHA_HYBRID = 0.7
ALPHA_GRID = [0.2, 0.4, 0.6, 0.8]
RUN_ALPHA_GRID = False  # set True for quick alpha sweep

BATCH_SIZE = 64
QWEN_BATCH_SIZE = 16
QWEN_MAX_LENGTH = 256

K_LIST = [5, 10, 20]
EVAL_MAX_USERS = None  # set to None for full evaluation

TAGS_LOWER = True
TAGS_DEDUP = True
TAGS_MAX_COUNT = 30
TAGS_MAX_LEN = 200
USE_GENOME_TAGS = False  # optional enrichment
GENOME_TAGS_TOPN = 5

EMBEDDING_MODELS = {
    "bert-all":  "sentence-transformers/all-bert-base-v2",
    "miniLM":    "sentence-transformers/all-MiniLM-L6-v2",
    "mpnet":     "sentence-transformers/all-mpnet-base-v2",
    "bge-small": "BAAI/bge-small-en-v1.5",
}

ACTIVE_MODELS = ["miniLM", "mpnet", "bge-small"]  # edit to select which models to run
RUN_QWEN = False  # set True only for a dedicated GPU run
QWEN_MODEL_NAME = "Qwen/Qwen-Embedding-0.6B"

USE_DISK_CACHE = True  # set True to reuse embeddings across runs
CACHE_DIR = "embeddings_cache"

RUN_POPULARITY_BASELINE = True
RUN_CORNAC_BASELINES = True  # set True for MF/BPR comparison
CORNAC_MODELS = ["mf", "bpr"]
CORNAC_FACTORS = 64
CORNAC_MAX_ITER = 50
CORNAC_LR = 0.01
CORNAC_REG = 0.02
CORNAC_MAX_USERS = None

pop_score_by_idx = None
LAST_EVAL_USERS = 0
RUN_CONFIG = {}

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda


Nous centralisons ici tous les hyperparamètres et “switches” du notebook afin de (1) faciliter les itérations, (2) garantir la reproductibilité, et (3) pouvoir exporter une configuration de run claire (run_config.json).

1) Reproductibilité et seeds
- RANDOM_STATE = 42 : graine unique pour stabiliser les splits et certains tirages aléatoires.
- np.random.seed / torch.manual_seed (+ cuda) : même objectif côté NumPy et PyTorch.
- DEVICE : sélection automatique CPU/GPU selon disponibilité.

2) Filtrage du dataset (réduction contrôlée du volume)
- MIN_USER_RATINGS = 15 : nous conservons uniquement les utilisateurs avec un minimum d’historique (évite les profils trop courts).
- MIN_ITEM_RATINGS = 50 : nous conservons uniquement les films suffisamment notés (évite les items trop rares / bruités).
- MAX_INTERACTIONS = None : pas de sous-échantillonnage par défaut ; peut être activé si la RAM est limitée.

3) Définition du split train/test
- TEST_SIZE = 0.2 : proportion réservée au test (si split aléatoire).
- MIN_INTERACTIONS_PER_USER = 2 : contrainte minimale pour pouvoir scinder un utilisateur entre train et test.
- MIN_RATING_POSITIVE = 4.0 : seuil “like” (rating >= 4) pour définir les items pertinents.
- SPLIT_STRATEGY = "time" : split temporel par utilisateur (prioritaire pour éviter la fuite d’information) ; "random" sert de fallback.

4) Construction du profil utilisateur (content-based)
- USER_PROFILE_MODE = "mean" : profil = moyenne des embeddings des films aimés.
  Options :
  - "mean" : simple et robuste
  - "rating" : pondère par la note (importance)
  - "recency" : pondère par la récence (préférences récentes)
  - "rating_recency" : combine les deux
- RECENCY_LAMBDA = 0.01 : intensité de la décroissance temporelle (plus grand = plus de poids aux interactions récentes).

5) Hybridation (contenu + popularité)
- USE_HYBRID = True : active un score hybride.
- ALPHA_HYBRID = 0.7 : pondération du score final
  score = alpha * score_embeddings + (1 - alpha) * score_popularité
- ALPHA_GRID / RUN_ALPHA_GRID : permet un balayage rapide des alphas (optionnel).

6) Paramètres embeddings (performance / mémoire)
- BATCH_SIZE = 64 : batch pour Sentence-Transformers (compromis vitesse/RAM/VRAM).
- QWEN_* : réglages dédiés à Qwen (batch plus petit + longueur max), à activer uniquement sur une machine GPU adaptée.
- K_LIST = [5, 10, 20] : valeurs de K pour les métriques top-K.
- EVAL_MAX_USERS = None : évaluation sur tous les utilisateurs éligibles (mettre un entier pour un run bridé).

7) Préparation des tags (qualité du texte)
- TAGS_LOWER / TAGS_DEDUP : normalisation et déduplication des tags (réduit le bruit).
- TAGS_MAX_COUNT / TAGS_MAX_LEN : plafonds pour contrôler la taille du texte et éviter l’explosion mémoire.
- USE_GENOME_TAGS / GENOME_TAGS_TOPN : enrichissement optionnel avec les “genome tags” (désactivé par défaut).

8) Modèles d’embeddings évalués
- EMBEDDING_MODELS : dictionnaire “clé courte → nom HF”.
- ACTIVE_MODELS : liste des modèles réellement exécutés sur ce run (facile à modifier).
- RUN_QWEN / QWEN_MODEL_NAME : option avancée (désactivée par défaut).

9) Cache embeddings (accélération et stabilité)
- USE_DISK_CACHE = True, CACHE_DIR : réutilisation des embeddings d’un run à l’autre (gain massif de temps, plus stable pour comparer les variantes).

10) Baselines d’évaluation
- RUN_POPULARITY_BASELINE = True : baseline indispensable (souvent très forte sur MovieLens).
- RUN_CORNAC_BASELINES = True : active MF/BPR via Cornac pour une comparaison collaborative.
- CORNAC_* : hyperparamètres (facteurs, itérations, learning rate, régularisation).
- CORNAC_MAX_USERS = None : possibilité de brider l’entraînement Cornac si nécessaire.

11) Variables d’état internes (runtime)
- pop_score_by_idx : cache des scores de popularité (réutilisé dans l’hybride).
- LAST_EVAL_USERS : trace du nombre d’utilisateurs réellement évalués.
- RUN_CONFIG = {} : conteneur destiné à exporter la configuration exacte du run (auditabilité).

En résumé, cette cellule fixe clairement “ce que nous testons” (modèles, split, profil utilisateur, hybridation), “sur qu
::contentReference[oaicite:0]{index=0}


------

## Configuration globale


In [16]:
def optimize_ratings_df(ratings):
    # Keep only necessary columns and reduce memory usage.
    keep_cols = ["userId", "movieId", "rating"]
    has_ts = "timestamp" in ratings.columns
    if has_ts:
        keep_cols.append("timestamp")
    else:
        print("timestamp not found -> fallback random split")

    ratings = ratings[keep_cols].copy()
    ratings["userId"] = ratings["userId"].astype("int32")
    ratings["movieId"] = ratings["movieId"].astype("int32")
    ratings["rating"] = ratings["rating"].astype("float32")
    if has_ts:
        ts = ratings["timestamp"]
        # Handle numeric timestamps and datetime strings.
        ts_num = pd.to_numeric(ts, errors="coerce")
        if ts_num.notna().mean() >= 0.8:
            ratings["timestamp"] = ts_num.astype("Int64")
        else:
            ts_parsed = pd.to_datetime(ts, errors="coerce", utc=True)
            if ts_parsed.isna().all():
                print("timestamp parse failed -> fallback random split")
                ratings["timestamp"] = pd.Series([pd.NA] * len(ratings), dtype="Int64")
            else:
                ts_int = ts_parsed.values.astype("int64") // 10**9
                ts_series = pd.Series(ts_int, index=ts_parsed.index)
                ts_series[ts_parsed.isna()] = pd.NA
                ratings["timestamp"] = ts_series.astype("Int64")
    return ratings
def load_movielens_20m():
    # Charge les principaux fichiers MovieLens 20M via kagglehub.
    ratings = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "grouplens/movielens-20m-dataset",
        "rating.csv",
    )
    movies = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "grouplens/movielens-20m-dataset",
        "movie.csv",
    )
    tags = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "grouplens/movielens-20m-dataset",
        "tag.csv",
    )

    print("ratings :", ratings.shape)
    print("movies  :", movies.shape)
    print("tags    :", tags.shape)

    return ratings, movies, tags


def load_genome_data():
    if not USE_GENOME_TAGS:
        return None, None
    try:
        genome_tags = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            "grouplens/movielens-20m-dataset",
            "genome-tags.csv",
        )
        genome_scores = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            "grouplens/movielens-20m-dataset",
            "genome-scores.csv",
        )
        print("genome_tags   :", genome_tags.shape)
        print("genome_scores :", genome_scores.shape)
        return genome_tags, genome_scores
    except Exception as exc:
        print("Genome tags not available, skipping:", repr(exc))
        return None, None


def filter_ratings(
    ratings,
    min_user_ratings=MIN_USER_RATINGS,
    min_item_ratings=MIN_ITEM_RATINGS,
    max_interactions=MAX_INTERACTIONS,
    random_state=RANDOM_STATE,
):
    # Filter users/items with few ratings, then downsample if needed.
    user_counts = ratings["userId"].value_counts()
    item_counts = ratings["movieId"].value_counts()

    good_users = user_counts[user_counts >= min_user_ratings].index
    good_items = item_counts[item_counts >= min_item_ratings].index

    ratings_filt = ratings[
        ratings["userId"].isin(good_users)
        & ratings["movieId"].isin(good_items)
    ].copy()

    if max_interactions is not None and len(ratings_filt) > max_interactions:
        ratings_filt = ratings_filt.sample(n=max_interactions, random_state=random_state)

    print("Taille initiale :", ratings.shape)
    print("Taille filtree  :", ratings_filt.shape)

    return ratings_filt


def build_movie_text_df(movies, tags, ratings_small):
    # Build a movie -> text dataframe from title, genres, and tags.
    movie_ids_used = ratings_small["movieId"].unique()

    tags_small = tags[tags["movieId"].isin(movie_ids_used)].copy()
    tags_small = tags_small.dropna(subset=["tag"])
    tags_small["tag"] = tags_small["tag"].astype(str)
    if TAGS_LOWER:
        tags_small["tag"] = tags_small["tag"].str.lower()

    def _aggregate_tags(series):
        tags_list = [t for t in series if isinstance(t, str)]
        if TAGS_DEDUP:
            seen = set()
            tags_list = [t for t in tags_list if not (t in seen or seen.add(t))]
        if TAGS_MAX_COUNT is not None:
            tags_list = tags_list[:TAGS_MAX_COUNT]
        tags_text = " ".join(tags_list)
        if TAGS_MAX_LEN is not None and len(tags_text) > TAGS_MAX_LEN:
            tags_text = tags_text[:TAGS_MAX_LEN]
        return tags_text

    tags_agg = (
        tags_small.groupby("movieId")["tag"]
        .apply(_aggregate_tags)
        .reset_index()
        .rename(columns={"tag": "tags_text"})
    )

    movies_text = movies.merge(tags_agg, on="movieId", how="left")
    movies_text = movies_text[movies_text["movieId"].isin(movie_ids_used)].copy()
    movies_text = movies_text.drop_duplicates("movieId")

    genome_tags, genome_scores = load_genome_data()
    if genome_tags is not None and genome_scores is not None:
        genome_scores = genome_scores[["movieId", "tagId", "relevance"]].copy()
        genome_scores = genome_scores[genome_scores["movieId"].isin(movie_ids_used)]
        genome_scores = genome_scores.sort_values(
            ["movieId", "relevance"], ascending=[True, False]
        )
        genome_scores = genome_scores.groupby("movieId").head(GENOME_TAGS_TOPN)

        genome_tags = genome_tags[["tagId", "tag"]].copy()
        genome_tags = genome_tags.rename(columns={"tag": "genome_tag"})
        genome_scores = genome_scores.merge(genome_tags, on="tagId", how="left")

        genome_scores["genome_tag"] = genome_scores["genome_tag"].astype(str)
        if TAGS_LOWER:
            genome_scores["genome_tag"] = genome_scores["genome_tag"].str.lower()

        genome_agg = (
            genome_scores.groupby("movieId")["genome_tag"]
            .apply(_aggregate_tags)
            .reset_index()
            .rename(columns={"genome_tag": "genome_tags_text"})
        )
        movies_text = movies_text.merge(genome_agg, on="movieId", how="left")
    else:
        movies_text["genome_tags_text"] = ""

    def _extract_year(title):
        match = re.search(r"\((\d{4})\)", title)
        return match.group(1) if match else None

    def build_text(row):
        title = str(row["title"])
        genres = str(row["genres"]).replace("|", " ")
        tags_text = row.get("tags_text")
        if pd.isna(tags_text):
            tags_text = ""
        genome_text = row.get("genome_tags_text")
        if pd.isna(genome_text):
            genome_text = ""
        year = _extract_year(title)

        parts = [title]
        if year:
            parts.append(f"Year: {year}")
        parts.append(f"Genres: {genres}")
        parts.append(f"Tags: {tags_text}")
        if genome_text:
            parts.append(f"Genome tags: {genome_text}")
        return ". ".join(parts)

    movies_text["text"] = movies_text.apply(build_text, axis=1)

    print("Nombre de films utilises :", len(movies_text))
    return movies_text


def train_test_split_by_user(
    ratings_df,
    test_size=TEST_SIZE,
    min_interactions=MIN_INTERACTIONS_PER_USER,
    random_state=RANDOM_STATE,
):
    # Split by user: keep a share for test, rest for train.
    train_parts = []
    test_parts = []

    for _, df_u in ratings_df.groupby("userId"):
        if len(df_u) < min_interactions:
            train_parts.append(df_u)
            continue

        train_u, test_u = train_test_split(
            df_u, test_size=test_size, random_state=random_state
        )
        train_parts.append(train_u)
        test_parts.append(test_u)

    train_df = pd.concat(train_parts).reset_index(drop=True)
    if len(test_parts) > 0:
        test_df = pd.concat(test_parts).reset_index(drop=True)
    else:
        test_df = pd.DataFrame(columns=ratings_df.columns)

    print("Train size :", train_df.shape)
    print("Test size  :", test_df.shape)
    print("Users with test :", len(test_parts))
    return train_df, test_df


def train_test_split_by_user_time(
    ratings_df,
    test_size=TEST_SIZE,
    min_interactions=MIN_INTERACTIONS_PER_USER,
):
    # Split by user with temporal ordering.
    train_parts = []
    test_parts = []
    users_with_test = 0

    for _, df_u in ratings_df.groupby("userId"):
        if len(df_u) < min_interactions:
            train_parts.append(df_u)
            continue

        df_u = df_u.sort_values("timestamp")
        n = len(df_u)
        n_test = int(np.ceil(test_size * n))
        n_test = min(max(n_test, 1), n - 1)

        train_u = df_u.iloc[:-n_test]
        test_u = df_u.iloc[-n_test:]
        train_parts.append(train_u)
        test_parts.append(test_u)
        users_with_test += 1

    train_df = pd.concat(train_parts).reset_index(drop=True)
    if len(test_parts) > 0:
        test_df = pd.concat(test_parts).reset_index(drop=True)
    else:
        test_df = pd.DataFrame(columns=ratings_df.columns)

    print("Train size :", train_df.shape)
    print("Test size  :", test_df.shape)
    print("Users with test :", users_with_test)
    return train_df, test_df


def validate_data(ratings_small, movies_text):
    if ratings_small.empty:
        raise ValueError("No data after filtering. Lower the thresholds.")
    n_ratings_items = ratings_small["movieId"].nunique()
    n_text_items = movies_text["movieId"].nunique()
    if n_ratings_items != n_text_items:
        print("Warning: items in ratings_small:", n_ratings_items)
        print("Warning: items in movies_text:", n_text_items)


Ce bloc couvre toute la chaîne “data” du notebook : (1) charger MovieLens 20M, (2) réduire l’empreinte mémoire, (3) filtrer le dataset pour un protocole stable, (4) construire un champ texte par film, puis (5) créer un split train/test robuste (temporel si possible).

1) Optimisation mémoire des ratings (optimize_ratings_df)
Nous ne conservons que les colonnes utiles : userId, movieId, rating (et timestamp si présent), puis nous castons les types pour réduire la RAM :
- userId, movieId → int32
- rating → float32
- timestamp → Int64 (nullable), converti si possible en “epoch seconds”.

Si le timestamp est absent ou illisible, nous l’annonçons explicitement et nous basculons vers un split aléatoire (fallback), afin de ne pas bloquer l’exécution.

2) Chargement MovieLens 20M via kagglehub (load_movielens_20m)
Nous chargeons les trois fichiers principaux :
- ratings : interactions (userId, movieId, rating, timestamp)
- movies : métadonnées (movieId, title, genres)
- tags : tags utilisateurs (userId, movieId, tag, timestamp)
Nous affichons leurs dimensions pour valider rapidement le chargement.

3) Enrichissement optionnel : genome-tags / genome-scores (load_genome_data)
Si USE_GENOME_TAGS = True, nous tentons de charger les fichiers “genome” (tags normalisés + scores de pertinence).
En cas d’échec, nous n’arrêtions pas le run : nous loguons l’erreur et continuons sans genome (comportement robuste).

4) Filtrage + sous-échantillonnage contrôlé (filter_ratings)
Pour stabiliser l’évaluation et éviter les cas extrêmes, nous filtrons :
- utilisateurs avec au moins MIN_USER_RATINGS notations,
- films avec au moins MIN_ITEM_RATINGS notations.
Optionnellement, nous pouvons limiter le volume avec MAX_INTERACTIONS (utile en environnement contraint).
Nous affichons la taille avant/après filtrage pour tracer l’impact.

5) Construction du texte par film (build_movie_text_df)
Nous construisons un DataFrame “movieId → text” uniquement pour les films réellement utilisés dans ratings_small :
- agrégation des tags par film (nettoyage : lowercase, déduplication, plafonds de nombre et de longueur),
- intégration des genres (séparateur “|” → espaces),
- ajout optionnel des genome tags (top-N par pertinence),
- extraction de l’année depuis le titre si présente.
Objectif : produire un texte explicable (title + year + genres + tags [+ genome]) pour calculer des embeddings.

6) Split train/test par utilisateur (train_test_split_by_user / train_test_split_by_user_time)
Deux stratégies, selon la disponibilité et la qualité du timestamp :
- split aléatoire par utilisateur : train_test_split_by_user
- split temporel par utilisateur (recommandé) : train_test_split_by_user_time
Le split temporel réduit le risque de fuite d’information et simule mieux un scénario réel (prédire le futur à partir du passé).

7) Validation minimale (validate_data)
Nous vérifions deux points :
- le dataset filt
::contentReference[oaicite:0]{index=0}


-----

In [17]:
ratings, movies, tags = load_movielens_20m()

ratings = optimize_ratings_df(ratings)
movies["movieId"] = movies["movieId"].astype("int32")
tags["movieId"] = tags["movieId"].astype("int32")

ratings_small = filter_ratings(
    ratings,
    min_user_ratings=MIN_USER_RATINGS,
    min_item_ratings=MIN_ITEM_RATINGS,
    max_interactions=MAX_INTERACTIONS,
    random_state=RANDOM_STATE,
)
ratings_small = ratings_small.reset_index(drop=True)

movies_text = build_movie_text_df(movies, tags, ratings_small)
validate_data(ratings_small, movies_text)

HAS_TIMESTAMP = "timestamp" in ratings_small.columns and ratings_small["timestamp"].notna().any()
if SPLIT_STRATEGY == "time" and HAS_TIMESTAMP:
    print("Split strategy: time")
    train_ratings, test_ratings = train_test_split_by_user_time(
        ratings_small,
        test_size=TEST_SIZE,
        min_interactions=MIN_INTERACTIONS_PER_USER,
    )
    SPLIT_STRATEGY_USED = "time"
else:
    if SPLIT_STRATEGY == "time" and not HAS_TIMESTAMP:
        print("timestamp not found -> fallback random split")
    print("Split strategy: random")
    train_ratings, test_ratings = train_test_split_by_user(
        ratings_small,
        test_size=TEST_SIZE,
        min_interactions=MIN_INTERACTIONS_PER_USER,
        random_state=RANDOM_STATE,
    )
    SPLIT_STRATEGY_USED = "random"

print("Nb d'utilisateurs uniques (ratings_small):", ratings_small["userId"].nunique())
print("Nb de films uniques (ratings_small)      :", ratings_small["movieId"].nunique())
print("Nb de films dans movies_text             :", movies_text["movieId"].nunique())


/tmp/ipykernel_55/3487322039.py:33: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  ratings = kagglehub.load_dataset(
/tmp/ipykernel_55/3487322039.py:38: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  movies = kagglehub.load_dataset(
/tmp/ipykernel_55/3487322039.py:43: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  tags = kagglehub.load_dataset(


ratings : (20000263, 4)
movies  : (27278, 3)
tags    : (465564, 4)
Taille initiale : (20000263, 4)
Taille filtree  : (19847947, 4)
Nombre de films utilises : 10524
Split strategy: time
Train size : (15823869, 4)
Test size  : (4024078, 4)
Users with test : 138493
Nb d'utilisateurs uniques (ratings_small): 138493
Nb de films uniques (ratings_small)      : 10524
Nb de films dans movies_text             : 10524


Chargement + filtrage + préparation du texte + split train/test (temporal-first)

Ce bloc exécute le pipeline de préparation des données de bout en bout : chargement MovieLens 20M, réduction mémoire, filtrage, construction du texte par film, puis split train/test en privilégiant un split temporel (anti-fuite).

1) Chargement + optimisation mémoire
Nous chargeons ratings, movies et tags depuis Kaggle (MovieLens 20M), puis nous réduisons l’empreinte mémoire :
- ratings : conservation des colonnes utiles + cast (int32/float32) et normalisation de timestamp,
- movies/tags : movieId casté en int32 pour limiter la RAM et accélérer les jointures.

Sortie observée :
- ratings : 20,000,263 lignes
- movies  : 27,278 films
- tags    : 465,564 tags

Note : les warnings “kagglehub.load_dataset() deprecated” ne bloquent pas l’exécution ; ils indiquent simplement qu’il faudra migrer vers dataset_load() à terme.

2) Filtrage contrôlé du dataset (stabiliser l’évaluation)
Nous filtrons les interactions pour retirer les utilisateurs et films trop rares :
- MIN_USER_RATINGS = 15
- MIN_ITEM_RATINGS = 50
Résultat : (19,847,947 interactions) après filtrage (réduction modérée → dataset encore très riche).

3) Construction du texte par film (features content-based)
Nous construisons movies_text uniquement sur les films présents dans ratings_small :
- texte = title + genres + tags (et options de nettoyage : lower, dédup, plafonds).
Résultat : 10,524 films avec un champ texte prêt pour embeddings.

4) Validation de cohérence
Nous vérifions que le nombre de films dans ratings_small correspond au nombre de films dans movies_text.
Résultat : cohérent (10,524 vs 10,524), donc pas de perte d’items entre interactions et représentation textuelle.

5) Split train/test : priorité au temporel (anti-fuite)
Nous détectons la présence de timestamp exploitable (HAS_TIMESTAMP = True).
Comme SPLIT_STRATEGY="time", nous appliquons un split temporel par utilisateur :
- Train size : 15,823,869
- Test size  : 4,024,078
- Users with test : 138,493

Intérêt : ce split reproduit un scénario réaliste (prédire les interactions futures à partir de l’historique) et limite la fuite d’information, contrairement à un split aléatoire.

6) Récapitulatif des volumes utilisés
- Utilisateurs uniques : 138,493
- Films uniques        : 10,524
- Films textuels        : 10,524

En résumé, ce bloc met en place une base de données cohérente et suffisamment volumineuse, avec un split temporel robuste, prête pour l’évaluation des méthodes content-based (embeddings) et collaboratives (Cornac).


-----

In [18]:
def optimize_ratings_df(ratings):
    # Keep only necessary columns and reduce memory usage.
    keep_cols = ["userId", "movieId", "rating"]
    has_ts = "timestamp" in ratings.columns
    if has_ts:
        keep_cols.append("timestamp")
    else:
        print("timestamp not found -> fallback random split")

    ratings = ratings[keep_cols].copy()
    ratings["userId"] = ratings["userId"].astype("int32")
    ratings["movieId"] = ratings["movieId"].astype("int32")
    ratings["rating"] = ratings["rating"].astype("float32")
    if has_ts:
        ts = ratings["timestamp"]
        # Handle numeric timestamps and datetime strings.
        ts_num = pd.to_numeric(ts, errors="coerce")
        if ts_num.notna().mean() >= 0.8:
            ratings["timestamp"] = ts_num.astype("Int64")
        else:
            ts_parsed = pd.to_datetime(ts, errors="coerce", utc=True)
            if ts_parsed.isna().all():
                print("timestamp parse failed -> fallback random split")
                ratings["timestamp"] = pd.Series([pd.NA] * len(ratings), dtype="Int64")
            else:
                ts_int = ts_parsed.values.astype("int64") // 10**9
                ts_series = pd.Series(ts_int, index=ts_parsed.index)
                ts_series[ts_parsed.isna()] = pd.NA
                ratings["timestamp"] = ts_series.astype("Int64")
    return ratings
def load_movielens_20m():
    # Charge les principaux fichiers MovieLens 20M via kagglehub.
    ratings = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "grouplens/movielens-20m-dataset",
        "rating.csv",
    )
    movies = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "grouplens/movielens-20m-dataset",
        "movie.csv",
    )
    tags = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "grouplens/movielens-20m-dataset",
        "tag.csv",
    )

    print("ratings :", ratings.shape)
    print("movies  :", movies.shape)
    print("tags    :", tags.shape)

    return ratings, movies, tags


def load_genome_data():
    if not USE_GENOME_TAGS:
        return None, None
    try:
        genome_tags = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            "grouplens/movielens-20m-dataset",
            "genome-tags.csv",
        )
        genome_scores = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            "grouplens/movielens-20m-dataset",
            "genome-scores.csv",
        )
        print("genome_tags   :", genome_tags.shape)
        print("genome_scores :", genome_scores.shape)
        return genome_tags, genome_scores
    except Exception as exc:
        print("Genome tags not available, skipping:", repr(exc))
        return None, None


def filter_ratings(
    ratings,
    min_user_ratings=MIN_USER_RATINGS,
    min_item_ratings=MIN_ITEM_RATINGS,
    max_interactions=MAX_INTERACTIONS,
    random_state=RANDOM_STATE,
):
    # Filter users/items with few ratings, then downsample if needed.
    user_counts = ratings["userId"].value_counts()
    item_counts = ratings["movieId"].value_counts()

    good_users = user_counts[user_counts >= min_user_ratings].index
    good_items = item_counts[item_counts >= min_item_ratings].index

    ratings_filt = ratings[
        ratings["userId"].isin(good_users)
        & ratings["movieId"].isin(good_items)
    ].copy()

    if max_interactions is not None and len(ratings_filt) > max_interactions:
        ratings_filt = ratings_filt.sample(n=max_interactions, random_state=random_state)

    print("Taille initiale :", ratings.shape)
    print("Taille filtree  :", ratings_filt.shape)

    return ratings_filt


def build_movie_text_df(movies, tags, ratings_small):
    # Build a movie -> text dataframe from title, genres, and tags.
    movie_ids_used = ratings_small["movieId"].unique()

    tags_small = tags[tags["movieId"].isin(movie_ids_used)].copy()
    tags_small = tags_small.dropna(subset=["tag"])
    tags_small["tag"] = tags_small["tag"].astype(str)
    if TAGS_LOWER:
        tags_small["tag"] = tags_small["tag"].str.lower()

    def _aggregate_tags(series):
        tags_list = [t for t in series if isinstance(t, str)]
        if TAGS_DEDUP:
            seen = set()
            tags_list = [t for t in tags_list if not (t in seen or seen.add(t))]
        if TAGS_MAX_COUNT is not None:
            tags_list = tags_list[:TAGS_MAX_COUNT]
        tags_text = " ".join(tags_list)
        if TAGS_MAX_LEN is not None and len(tags_text) > TAGS_MAX_LEN:
            tags_text = tags_text[:TAGS_MAX_LEN]
        return tags_text

    tags_agg = (
        tags_small.groupby("movieId")["tag"]
        .apply(_aggregate_tags)
        .reset_index()
        .rename(columns={"tag": "tags_text"})
    )

    movies_text = movies.merge(tags_agg, on="movieId", how="left")
    movies_text = movies_text[movies_text["movieId"].isin(movie_ids_used)].copy()
    movies_text = movies_text.drop_duplicates("movieId")

    genome_tags, genome_scores = load_genome_data()
    if genome_tags is not None and genome_scores is not None:
        genome_scores = genome_scores[["movieId", "tagId", "relevance"]].copy()
        genome_scores = genome_scores[genome_scores["movieId"].isin(movie_ids_used)]
        genome_scores = genome_scores.sort_values(
            ["movieId", "relevance"], ascending=[True, False]
        )
        genome_scores = genome_scores.groupby("movieId").head(GENOME_TAGS_TOPN)

        genome_tags = genome_tags[["tagId", "tag"]].copy()
        genome_tags = genome_tags.rename(columns={"tag": "genome_tag"})
        genome_scores = genome_scores.merge(genome_tags, on="tagId", how="left")

        genome_scores["genome_tag"] = genome_scores["genome_tag"].astype(str)
        if TAGS_LOWER:
            genome_scores["genome_tag"] = genome_scores["genome_tag"].str.lower()

        genome_agg = (
            genome_scores.groupby("movieId")["genome_tag"]
            .apply(_aggregate_tags)
            .reset_index()
            .rename(columns={"genome_tag": "genome_tags_text"})
        )
        movies_text = movies_text.merge(genome_agg, on="movieId", how="left")
    else:
        movies_text["genome_tags_text"] = ""

    def _extract_year(title):
        match = re.search(r"\((\d{4})\)", title)
        return match.group(1) if match else None

    def build_text(row):
        title = str(row["title"])
        genres = str(row["genres"]).replace("|", " ")
        tags_text = row.get("tags_text")
        if pd.isna(tags_text):
            tags_text = ""
        genome_text = row.get("genome_tags_text")
        if pd.isna(genome_text):
            genome_text = ""
        year = _extract_year(title)

        parts = [title]
        if year:
            parts.append(f"Year: {year}")
        parts.append(f"Genres: {genres}")
        parts.append(f"Tags: {tags_text}")
        if genome_text:
            parts.append(f"Genome tags: {genome_text}")
        return ". ".join(parts)

    movies_text["text"] = movies_text.apply(build_text, axis=1)

    print("Nombre de films utilises :", len(movies_text))
    return movies_text


def train_test_split_by_user(
    ratings_df,
    test_size=TEST_SIZE,
    min_interactions=MIN_INTERACTIONS_PER_USER,
    random_state=RANDOM_STATE,
):
    # Split by user: keep a share for test, rest for train.
    train_parts = []
    test_parts = []

    for _, df_u in ratings_df.groupby("userId"):
        if len(df_u) < min_interactions:
            train_parts.append(df_u)
            continue

        train_u, test_u = train_test_split(
            df_u, test_size=test_size, random_state=random_state
        )
        train_parts.append(train_u)
        test_parts.append(test_u)

    train_df = pd.concat(train_parts).reset_index(drop=True)
    if len(test_parts) > 0:
        test_df = pd.concat(test_parts).reset_index(drop=True)
    else:
        test_df = pd.DataFrame(columns=ratings_df.columns)

    print("Train size :", train_df.shape)
    print("Test size  :", test_df.shape)
    print("Users with test :", len(test_parts))
    return train_df, test_df


def train_test_split_by_user_time(
    ratings_df,
    test_size=TEST_SIZE,
    min_interactions=MIN_INTERACTIONS_PER_USER,
):
    # Split by user with temporal ordering.
    train_parts = []
    test_parts = []
    users_with_test = 0

    for _, df_u in ratings_df.groupby("userId"):
        if len(df_u) < min_interactions:
            train_parts.append(df_u)
            continue

        df_u = df_u.sort_values("timestamp")
        n = len(df_u)
        n_test = int(np.ceil(test_size * n))
        n_test = min(max(n_test, 1), n - 1)

        train_u = df_u.iloc[:-n_test]
        test_u = df_u.iloc[-n_test:]
        train_parts.append(train_u)
        test_parts.append(test_u)
        users_with_test += 1

    train_df = pd.concat(train_parts).reset_index(drop=True)
    if len(test_parts) > 0:
        test_df = pd.concat(test_parts).reset_index(drop=True)
    else:
        test_df = pd.DataFrame(columns=ratings_df.columns)

    print("Train size :", train_df.shape)
    print("Test size  :", test_df.shape)
    print("Users with test :", users_with_test)
    return train_df, test_df


def validate_data(ratings_small, movies_text):
    if ratings_small.empty:
        raise ValueError("No data after filtering. Lower the thresholds.")
    n_ratings_items = ratings_small["movieId"].nunique()
    n_text_items = movies_text["movieId"].nunique()
    if n_ratings_items != n_text_items:
        print("Warning: items in ratings_small:", n_ratings_items)
        print("Warning: items in movies_text:", n_text_items)


Ce bloc regroupe les fonctions cœur du pipeline de préparation des données. L’objectif est de (i) charger MovieLens 20M proprement, (ii) réduire la consommation mémoire, (iii) construire une représentation textuelle par film (pour les embeddings), puis (iv) produire un split train/test robuste, idéalement temporel (anti-fuite).

1) optimize_ratings_df(ratings) — optimisation RAM + normalisation timestamp
Nous réduisons l’empreinte mémoire du fichier ratings :
- conservation des colonnes utiles : userId, movieId, rating (+ timestamp si présent),
- cast en int32 / float32,
- normalisation du timestamp :
  - si majoritairement numérique : conversion en entier nullable (Int64),
  - sinon : tentative de parsing datetime → conversion en epoch seconds,
  - en cas d’échec : timestamp mis à NA et bascule implicite vers split aléatoire.

Intérêt : cette étape est critique pour éviter les crash RAM et garantir que le split temporel fonctionne lorsque possible.

2) load_movielens_20m() — chargement des fichiers principaux via kagglehub
Nous chargeons les trois fichiers essentiels du projet :
- rating.csv : interactions (userId, movieId, rating, timestamp),
- movie.csv  : métadonnées film (movieId, title, genres),
- tag.csv    : tags utilisateurs (userId, movieId, tag, timestamp).
La fonction affiche les shapes pour un contrôle rapide de l’intégrité du chargement.

3) load_genome_data() — enrichissement optionnel (genome tags)
Si USE_GENOME_TAGS=True, nous tentons de charger :
- genome-tags.csv (vocabulaire normalisé),
- genome-scores.csv (pertinence tagId ↔ film).
En cas d’indisponibilité ou d’erreur, nous continuons sans bloquer (fallback propre).
Intérêt : enrichir le texte des films avec des tags normalisés (signal plus stable que les tags libres).

4) filter_ratings(...) — filtrage par fréquence + sous-échantillonnage optionnel
Nous stabilisons le dataset en retirant les entités trop rares :
- utilisateurs avec moins de MIN_USER_RATINGS interactions,
- films avec moins de MIN_ITEM_RATINGS interactions,
- et optionnellement un downsample à MAX_INTERACTIONS pour itérations rapides.
La fonction affiche la taille initiale et la taille filtrée pour tracer l’impact du filtrage.

5) build_movie_text_df(movies, tags, ratings_small) — construction du texte par film
Nous construisons un dataframe “movieId → text” uniquement sur les films effectivement présents dans ratings_small :
- agrégation des tags par film (nettoyage : lower, déduplication, plafonds de nombre et de longueur),
- fusion movies + tags agrégés,
- ajout optionnel des genome tags (top-N par pertinence),
- génération d’un texte explicable et stable, du type :
  "Title. Year: YYYY. Genres: ... Tags: ... Genome tags: ..."
Résultat : une représentation textuelle prête à encoder par Sentence-Transformers.

Intérêt : standardiser l’entrée “contenu” du modèle et contrôler la variabilité des tags (sinon explosion mémoire/bruit).

6) train_test_split_by_user(...) — split aléatoire par utilisateur (fallback)
Split par utilisateur quand le temporel n’est pas disponible :
- pour chaque utilisateur, on réserve test_size en test (train_test_split),
- garantie minimale d’interactions (min_interactions),
- maintien d’un test par utilisateur quand possible.
Intérêt : garder une évaluation par utilisateur même en absence de timestamp fiable.

7) train_test_split_by_user_time(...) — split temporel par utilisateur (recommandé)
Split “réaliste” :
- tri des interactions par timestamp pour chaque utilisateur,
- conservation des dernières interactions en test,
- le reste en train.
Intérêt : anti-fuite et alignement avec un cas réel (prédiction du futur à partir du passé).

8) validate_data(ratings_small, movies_text) — garde-fou de cohérence
Nous vérifions :
- que le dataset filtré n’est pas vide (sinon seuils trop stricts),
- que le nombre de films dans ratings_small est bien égal au nombre de films encodables dans movies_text.
Intérêt : éviter une évaluation biaisée (items sans texte) et diagnostiquer rapidement une perte d’items due aux jointures.

En synthèse, ces fonctions rendent le pipeline plus robuste (RAM), plus propre (texte contrôlé), et plus fiable (split temporel + validation), ce qui est indispensable avant de comparer embeddings vs baselines collaboratives.


-----

## Chargement et preparation des donnees


In [19]:
def get_cache_path(model_key):
    if not USE_DISK_CACHE:
        return None
    os.makedirs(CACHE_DIR, exist_ok=True)
    return os.path.join(CACHE_DIR, f"{model_key}.npy")


def load_embeddings_cache(cache_path):
    if cache_path and os.path.exists(cache_path):
        emb = np.load(cache_path)
        print("Loaded embeddings:", cache_path, emb.shape)
        return emb
    return None


def save_embeddings_cache(cache_path, emb):
    if cache_path:
        np.save(cache_path, emb)


def compute_item_embeddings_st(model_path, texts, batch_size=BATCH_SIZE, device=DEVICE):
    # Compute normalized item embeddings with SentenceTransformers.
    print(f"=== Model: {model_path} ===")
    model = SentenceTransformer(model_path, device=device)
    emb = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    emb = emb.astype("float32", copy=False)
    print("Embeddings shape:", emb.shape)
    return emb


def compute_item_embeddings_qwen(
    model_name,
    texts,
    batch_size=QWEN_BATCH_SIZE,
    max_length=QWEN_MAX_LENGTH,
    device=DEVICE,
):
    # Compute normalized item embeddings using a Qwen embedding model.
    print(f"=== Qwen model: {model_name} ===")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
    model.to(device)
    model.eval()

    all_embeddings = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start : start + batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}
            outputs = model(**encoded)

            token_embeddings = outputs.last_hidden_state
            attention_mask = encoded["attention_mask"].unsqueeze(-1)
            masked = token_embeddings * attention_mask
            summed = masked.sum(dim=1)
            counts = attention_mask.sum(dim=1).clamp(min=1e-9)
            pooled = summed / counts
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            all_embeddings.append(pooled.cpu().numpy())

    emb = np.vstack(all_embeddings).astype("float32", copy=False)
    print("Embeddings shape:", emb.shape)
    return emb


def build_user_item_dict(ratings_df, min_rating_positive=None):
    # Map user -> set of movieIds; optionally keep only positives.
    if min_rating_positive is None:
        df = ratings_df
    else:
        df = ratings_df[ratings_df["rating"] >= min_rating_positive]
    return df.groupby("userId")["movieId"].apply(set).to_dict()


def build_user_interactions(ratings_df, min_rating_positive=None):
    # Map user -> list of (movieId, rating, timestamp).
    if min_rating_positive is None:
        df = ratings_df
    else:
        df = ratings_df[ratings_df["rating"] >= min_rating_positive]

    has_ts = "timestamp" in df.columns
    cols = ["userId", "movieId", "rating"] + (["timestamp"] if has_ts else [])
    df = df[cols].copy()
    if has_ts:
        df = df.sort_values(["userId", "timestamp"])

    if has_ts:
        grouped = (
            df.groupby("userId")[["movieId", "rating", "timestamp"]]
            .apply(lambda x: list(map(tuple, x.values)))
            .to_dict()
        )
    else:
        grouped = (
            df.groupby("userId")[["movieId", "rating"]]
            .apply(lambda x: [(int(a), float(b), None) for a, b in x.values])
            .to_dict()
        )
    return grouped


def build_user_embeddings(
    train_user_data,
    item_embeddings,
    movieid2idx,
    user_profile_mode=USER_PROFILE_MODE,
    recency_lambda=RECENCY_LAMBDA,
):
    # Build user embeddings with optional rating/recency weighting.
    if not hasattr(build_user_embeddings, "_missing_ts_warned"):
        build_user_embeddings._missing_ts_warned = False

    user_embeddings = {}
    for uid, items in train_user_data.items():
        if not items:
            continue

        is_plain_ids = isinstance(items, set)
        if isinstance(items, list) and items:
            is_plain_ids = is_plain_ids or isinstance(items[0], (int, np.integer))

        if is_plain_ids:
            movie_ids = list(items)
            idxs = [movieid2idx[mid] for mid in movie_ids if mid in movieid2idx]
            if not idxs:
                continue
            emb = item_embeddings[idxs].mean(axis=0)
        else:
            idxs = []
            ratings = []
            timestamps = []
            for entry in items:
                if len(entry) >= 3:
                    mid, rating, ts = entry[0], entry[1], entry[2]
                else:
                    mid, rating, ts = entry[0], entry[1], None

                if ts is not None and pd.isna(ts):
                    ts = None

                if mid not in movieid2idx:
                    continue
                idxs.append(movieid2idx[mid])
                ratings.append(float(rating))
                timestamps.append(ts)

            if not idxs:
                continue

            mode = user_profile_mode
            ts_vals = [t for t in timestamps if t is not None]
            t_max = max(ts_vals) if ts_vals else None
            if mode in ("recency", "rating_recency") and t_max is None:
                if not build_user_embeddings._missing_ts_warned:
                    print("Recency mode requested but timestamp missing -> fallback")
                    build_user_embeddings._missing_ts_warned = True
                mode = "rating" if mode == "rating_recency" else "mean"

            if mode == "mean":
                emb = item_embeddings[idxs].mean(axis=0)
            else:
                weights = []
                for rating, ts in zip(ratings, timestamps):
                    w = 1.0
                    if mode in ("rating", "rating_recency"):
                        w *= rating
                    if mode in ("recency", "rating_recency"):
                        age_seconds = float(t_max - ts) if ts is not None else 0.0
                        age = age_seconds / 86400.0
                        w *= float(np.exp(-recency_lambda * age))
                    weights.append(w)
                weights = np.asarray(weights, dtype="float32")
                if weights.sum() <= 0:
                    emb = item_embeddings[idxs].mean(axis=0)
                else:
                    emb = np.average(item_embeddings[idxs], axis=0, weights=weights)

        norm = np.linalg.norm(emb)
        if norm > 0:
            emb = emb / norm
        user_embeddings[uid] = emb

    return user_embeddings


def build_popularity_scores(train_ratings, movieid2idx):
    # Log-count popularity score aligned with item indices.
    counts = train_ratings["movieId"].value_counts()
    pop_score = np.zeros(len(movieid2idx), dtype="float32")
    for mid, count in counts.items():
        idx = movieid2idx.get(int(mid))
        if idx is not None:
            pop_score[idx] = np.log1p(count)

    min_v = float(pop_score.min())
    max_v = float(pop_score.max())
    if max_v > min_v:
        pop_score = (pop_score - min_v) / (max_v - min_v)
    return pop_score


def recommend_for_user(
    user_id,
    user_embedding,
    item_embeddings,
    train_user_items_all,
    movieid2idx,
    idx2movieid,
    top_k=10,
):
    # Recommend top_k items by cosine similarity, excluding seen items.
    scores = item_embeddings @ user_embedding

    if USE_HYBRID:
        if pop_score_by_idx is None:
            if not hasattr(recommend_for_user, "_hybrid_warned"):
                print("USE_HYBRID=True but pop_score_by_idx is None -> content-only")
                recommend_for_user._hybrid_warned = True
        elif len(pop_score_by_idx) == len(scores):
            scores = ALPHA_HYBRID * scores + (1.0 - ALPHA_HYBRID) * pop_score_by_idx
        else:
            if not hasattr(recommend_for_user, "_hybrid_len_warned"):
                print("pop_score_by_idx length mismatch -> content-only")
                recommend_for_user._hybrid_len_warned = True

    seen = train_user_items_all.get(user_id, set())
    if seen:
        seen_idx = [movieid2idx[mid] for mid in seen if mid in movieid2idx]
        scores[seen_idx] = -np.inf

    if top_k >= len(scores):
        top_idx = np.argsort(scores)[::-1]
    else:
        top_idx = np.argpartition(scores, -top_k)[-top_k:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

    return [idx2movieid[idx] for idx in top_idx]


def recall_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0
    rec_k = recommended[:k]
    return len(set(rec_k) & relevant) / len(relevant)


def ndcg_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    dcg = 0.0
    for i, item in enumerate(rec_k):
        if item in relevant:
            dcg += 1.0 / np.log2(i + 2)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0


def map_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0
    rec_k = recommended[:k]
    hits = 0
    ap = 0.0
    for i, item in enumerate(rec_k, start=1):
        if item in relevant:
            hits += 1
            ap += hits / i
    return ap / min(len(relevant), k)


def hit_rate_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    return 1.0 if len(set(rec_k) & relevant) > 0 else 0.0


def get_eval_users(
    train_user_data,
    test_user_items,
    max_users=EVAL_MAX_USERS,
    random_state=RANDOM_STATE,
):
    users = [u for u in train_user_data if u in test_user_items and len(test_user_items[u]) > 0]
    if max_users is not None and len(users) > max_users:
        rng = np.random.default_rng(random_state)
        users = rng.choice(users, size=max_users, replace=False)
    return list(users)


def evaluate_model(
    item_embeddings,
    train_user_interactions,
    train_user_items_all,
    test_user_items,
    movieid2idx,
    idx2movieid,
    k_list,
    max_users=EVAL_MAX_USERS,
    random_state=RANDOM_STATE,
):
    global LAST_EVAL_USERS

    user_embeddings = build_user_embeddings(
        train_user_interactions, item_embeddings, movieid2idx
    )

    users = get_eval_users(
        train_user_interactions, test_user_items, max_users=max_users, random_state=random_state
    )
    users = [u for u in users if u in user_embeddings]

    LAST_EVAL_USERS = len(users)
    print("Users eval:", len(users))

    metrics = {f"Recall@{k}": [] for k in k_list}
    metrics.update({f"NDCG@{k}": [] for k in k_list})
    metrics.update({f"MAP@{k}": [] for k in k_list})
    metrics.update({f"HR@{k}": [] for k in k_list})

    max_k = max(k_list)
    for uid in users:
        recs = recommend_for_user(
            uid,
            user_embeddings[uid],
            item_embeddings,
            train_user_items_all,
            movieid2idx,
            idx2movieid,
            top_k=max_k,
        )
        relevant = test_user_items[uid]

        for k in k_list:
            metrics[f"Recall@{k}"].append(recall_at_k(recs, relevant, k))
            metrics[f"NDCG@{k}"].append(ndcg_at_k(recs, relevant, k))
            metrics[f"MAP@{k}"].append(map_at_k(recs, relevant, k))
            metrics[f"HR@{k}"].append(hit_rate_at_k(recs, relevant, k))

    return {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()}


def build_popularity_ranking(train_ratings):
    return train_ratings["movieId"].value_counts().index.tolist()


def evaluate_popularity_baseline(
    popular_items,
    train_user_items_all,
    test_user_items,
    k_list,
    max_users=EVAL_MAX_USERS,
    random_state=RANDOM_STATE,
):
    users = [u for u in test_user_items if u in train_user_items_all and len(test_user_items[u]) > 0]
    if max_users is not None and len(users) > max_users:
        rng = np.random.default_rng(random_state)
        users = rng.choice(users, size=max_users, replace=False)

    print("Users eval (popularity):", len(users))

    metrics = {f"Recall@{k}": [] for k in k_list}
    metrics.update({f"NDCG@{k}": [] for k in k_list})
    metrics.update({f"MAP@{k}": [] for k in k_list})
    metrics.update({f"HR@{k}": [] for k in k_list})

    max_k = max(k_list)
    for uid in users:
        seen = train_user_items_all.get(uid, set())
        recs = []
        for mid in popular_items:
            if mid in seen:
                continue
            recs.append(mid)
            if len(recs) >= max_k:
                break

        relevant = test_user_items[uid]
        for k in k_list:
            metrics[f"Recall@{k}"].append(recall_at_k(recs, relevant, k))
            metrics[f"NDCG@{k}"].append(ndcg_at_k(recs, relevant, k))
            metrics[f"MAP@{k}"].append(map_at_k(recs, relevant, k))
            metrics[f"HR@{k}"].append(hit_rate_at_k(recs, relevant, k))

    return {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()}


def score_all_items_cornac(model, user_idx):
    if hasattr(model, "score"):
        try:
            scores = model.score(user_idx)
            scores = np.asarray(scores)
            if scores.ndim == 1:
                return scores
        except Exception:
            pass

    if hasattr(model, "U") and hasattr(model, "V"):
        scores = model.U[user_idx] @ model.V.T
        if hasattr(model, "bu"):
            scores = scores + model.bu[user_idx]
        if hasattr(model, "bi"):
            scores = scores + model.bi
        return np.asarray(scores)

    raise RuntimeError("Cornac model does not support score-all-items")


def evaluate_cornac_model(
    model,
    train_set,
    train_user_items_all,
    test_user_items,
    k_list,
    max_users=EVAL_MAX_USERS,
    random_state=RANDOM_STATE,
):
    uid_map = train_set.uid_map
    iid_map = train_set.iid_map
    idx2item = {idx: item for item, idx in iid_map.items()}

    users = [u for u in test_user_items if u in uid_map and len(test_user_items[u]) > 0]
    if max_users is not None and len(users) > max_users:
        rng = np.random.default_rng(random_state)
        users = rng.choice(users, size=max_users, replace=False)

    print("Users eval (cornac):", len(users))

    metrics = {f"Recall@{k}": [] for k in k_list}
    metrics.update({f"NDCG@{k}": [] for k in k_list})
    metrics.update({f"MAP@{k}": [] for k in k_list})
    metrics.update({f"HR@{k}": [] for k in k_list})

    max_k = max(k_list)
    for uid in users:
        uidx = uid_map[uid]
        scores = score_all_items_cornac(model, uidx)
        scores = np.array(scores, dtype="float32", copy=True)

        seen = train_user_items_all.get(uid, set())
        if seen:
            seen_idx = [iid_map[mid] for mid in seen if mid in iid_map]
            scores[seen_idx] = -np.inf

        if max_k >= len(scores):
            top_idx = np.argsort(scores)[::-1]
        else:
            top_idx = np.argpartition(scores, -max_k)[-max_k:]
            top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

        recs = [idx2item[idx] for idx in top_idx]
        relevant = test_user_items[uid]

        for k in k_list:
            metrics[f"Recall@{k}"].append(recall_at_k(recs, relevant, k))
            metrics[f"NDCG@{k}"].append(ndcg_at_k(recs, relevant, k))
            metrics[f"MAP@{k}"].append(map_at_k(recs, relevant, k))
            metrics[f"HR@{k}"].append(hit_rate_at_k(recs, relevant, k))

    return {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()}


def inspect_user(user_id, model_key, top_n=10):
    # Quick qualitative inspection: liked items vs top-N recommendations.
    if model_key in ("qwen", "qwen-0.6B"):
        model_key = "qwen-0.6B"

    if user_id not in train_user_items_all:
        print("Unknown user_id:", user_id)
        return

    if model_key not in item_embeddings_cache:
        if model_key in EMBEDDING_MODELS:
            print("Computing embeddings for:", model_key)
            cache_path = get_cache_path(model_key)
            emb = load_embeddings_cache(cache_path)
            if emb is None:
                emb = compute_item_embeddings_st(
                    EMBEDDING_MODELS[model_key],
                    texts,
                    batch_size=BATCH_SIZE,
                    device=DEVICE,
                )
                save_embeddings_cache(cache_path, emb)
            item_embeddings_cache[model_key] = emb
        elif model_key == "qwen-0.6B":
            if not RUN_QWEN:
                print("RUN_QWEN=False; set it True and run Qwen embeddings first.")
                return
            cache_path = get_cache_path(model_key)
            emb = load_embeddings_cache(cache_path)
            if emb is None:
                emb = compute_item_embeddings_qwen(
                    QWEN_MODEL_NAME,
                    texts,
                    batch_size=QWEN_BATCH_SIZE,
                    max_length=QWEN_MAX_LENGTH,
                    device=DEVICE,
                )
                save_embeddings_cache(cache_path, emb)
            item_embeddings_cache[model_key] = emb
        else:
            print("Unknown model_key:", model_key)
            return

    interactions = train_user_interactions.get(user_id, [])
    if not interactions:
        print("No interactions for user:", user_id)
        return

    user_emb = build_user_embeddings(
        {user_id: interactions},
        item_embeddings_cache[model_key],
        movieid2idx,
    ).get(user_id)
    if user_emb is None:
        print("Cannot build user embedding for:", user_id)
        return

    recs = recommend_for_user(
        user_id,
        user_emb,
        item_embeddings_cache[model_key],
        train_user_items_all,
        movieid2idx,
        idx2movieid,
        top_k=top_n,
    )

    def _format_titles(ids, limit=10):
        titles = [movieid2title.get(mid, str(mid)) for mid in ids]
        if len(titles) > limit:
            return titles[:limit] + ["..."]
        return titles

    liked = train_user_items_pos.get(user_id, set())
    print("User:", user_id)
    print("Train likes:", _format_titles(sorted(liked), limit=top_n))
    test_likes = test_user_items.get(user_id, set())
    if test_likes:
        print("Test likes :", _format_titles(sorted(test_likes), limit=top_n))
    print("Top-N recs :", _format_titles(recs, limit=top_n))


Ce bloc implémente la partie “modélisation + évaluation” du projet. Il couvre : (i) le calcul des embeddings items (Sentence-Transformers ou Qwen), (ii) un cache disque pour éviter de recalculer, (iii) la construction des profils utilisateurs, (iv) la recommandation top-K (avec option hybride popularité), (v) les métriques (Recall/NDCG/MAP/HR), (vi) l’évaluation sur un ensemble d’utilisateurs, et (vii) les baselines Popularité et Cornac (MF/BPR). L’objectif est d’obtenir des résultats reproductibles et comparables, sans exploser le temps de calcul.

1) Cache disque des embeddings (get_cache_path / load_embeddings_cache / save_embeddings_cache)
Nous stockons les embeddings calculés dans CACHE_DIR sous forme de .npy :
- si le fichier existe : on recharge immédiatement (gain de temps majeur),
- sinon : on calcule puis on sauvegarde.
Intérêt : itérer rapidement sur les étapes aval (profil utilisateur, métriques) sans recalculer les embeddings à chaque exécution.

2) Embeddings items – Sentence-Transformers (compute_item_embeddings_st)
Nous calculons un embedding par film à partir du texte (title + genres + tags) :
- encodage par batch pour contrôler la RAM/VRAM,
- normalize_embeddings=True pour obtenir des vecteurs déjà normalisés (cosine similarity = simple produit scalaire),
- cast float32 pour réduire l’empreinte mémoire.
Intérêt : approche content-based, interprétable, et rapide à prototyper.

3) Embeddings items – Qwen (compute_item_embeddings_qwen)
Option GPU dédiée (plus lourde) :
- tokenisation (padding/truncation),
- passage dans le modèle,
- mean pooling masqué par attention_mask,
- normalisation L2.
Intérêt : tester un encodeur plus puissant, au prix de ressources nettement supérieures (batch_size plus faible, max_length contrôlé).

4) Structures utilisateur/items (build_user_item_dict / build_user_interactions)
Nous construisons des structures utiles et rapides :
- user → set(movieId) pour exclure les films déjà vus au moment de recommander,
- user → liste (movieId, rating, timestamp) pour construire des profils pondérés (rating/recency).
Intérêt : séparation claire entre (a) exclusion “seen items” et (b) construction de profil.

5) Profils utilisateurs (build_user_embeddings)
Nous construisons un embedding utilisateur en agrégeant les embeddings des films du train, avec 4 modes :
- mean : moyenne simple,
- rating : pondération par la note,
- recency : pondération par la fraîcheur (décroissance exponentielle),
- rating_recency : combinaison des deux.
Si le mode recency est demandé mais que timestamp est manquant, nous basculons proprement (fallback) et on log une seule fois.
Intérêt : personnalisation contrôlée, explicable, et paramétrable.

6) Popularité (build_popularity_scores)
Nous calculons un score de popularité par film (log1p(count)) puis min-max scaling.
Ce score sert à :
- une baseline “popularité pure” (ranking global),
- une hybridation avec le score content-based (voir recommend_for_user).
Intérêt : baseline très compétitive sur MovieLens et signal robuste pour hybrider.

7) Recommandation top-K (recommend_for_user)
Nous scorons tous les films par produit scalaire item_embeddings @ user_embedding (car vecteurs normalisés).
Ensuite :
- option hybride (USE_HYBRID) :
  score_final = ALPHA_HYBRID * score_content + (1-ALPHA_HYBRID) * score_popularité
- exclusion stricte des items déjà vus (scores mis à -inf),
- extraction top-K via argpartition (plus rapide que trier tout).
Intérêt : recommandation efficace, sans fuite (pas de “déjà vus”), et paramétrable.

8) Métriques top-K (recall_at_k / ndcg_at_k / map_at_k / hit_rate_at_k)
Nous reportons 4 métriques complémentaires :
- Recall@K : couverture des items pertinents,
- NDCG@K : qualité de l’ordre (hits en haut = mieux),
- MAP@K : précision cumulée moyenne,
- HR@K : “au moins un hit” dans le top-K.
Intérêt : lecture plus complète qu’un seul score.

9) Sélection des utilisateurs d’évaluation (get_eval_users)
Nous évaluons uniquement les utilisateurs qui :
- ont des interactions en train (profil possible),
- ont au moins 1 item pertinent en test.
Option : sous-échantillonnage EVAL_MAX_USERS pour des runs rapides.

10) Évaluation modèle embeddings (evaluate_model)
Pipeline complet :
- construction des user_embeddings,
- boucle sur les users,
- recommandation top-max(K_LIST),
- calcul et moyenne des métriques.
LAST_EVAL_USERS mémorise le nombre d’utilisateurs effectivement évalués.
Intérêt : procédure unique, stable, comparable entre modèles.

11) Baseline popularité (evaluate_popularity_baseline)
Nous recommandons le ranking global popular_items en excluant les films vus.
Cela donne une baseline simple mais difficile à battre sur MovieLens.
Intérêt : point de repère indispensable.

12) Baselines Cornac (evaluate_cornac_model + score_all_items_cornac)
Nous évaluons MF/BPR via Cornac :
- extraction des scores “all items” (méthode score() si dispo, sinon U@V^T),
- exclusion des items vus,
- top-K et métriques identiques aux autres approches.
Intérêt : comparaison directe content-based vs collaboratif, à protocole égal.

13) Inspection qualitative (inspect_user)
Outil de sanity-check :
- affiche les films “aimés” en train, les films pertinents en test (si dispo),
- affiche les recommandations top-N (avec titres via movieid2title).
Intérêt : vérifier que le système recommande des films plausibles (au-delà des métriques).

En synthèse, ce bloc fournit un pipeline complet, reproductible et comparable :
- embeddings → profils utilisateurs → recommandations top-K → métriques,
- + baselines (popularité, Cornac) et inspection qualitative,
tout en gardant les coûts sous contrôle via cache disque e
::contentReference[oaicite:0]{index=0}


------

In [20]:
movies_text = movies_text.sort_values("movieId").reset_index(drop=True)

movie_ids = movies_text["movieId"].tolist()
texts = movies_text["text"].tolist()

movieid2idx = {mid: idx for idx, mid in enumerate(movie_ids)}
idx2movieid = {idx: mid for mid, idx in movieid2idx.items()}
movieid2title = dict(zip(movies_text["movieId"], movies_text["title"]))

print("Nb de films dans movie_ids :", len(movie_ids))
print("Nb de textes               :", len(texts))


Nb de films dans movie_ids : 10524
Nb de textes               : 10524


Nous préparons ici les structures d’indexation nécessaires pour calculer les embeddings des films et assurer une correspondance fiable entre :
- les identifiants MovieLens (movieId),
- les positions dans les matrices (index),
- les titres (pour l’inspection qualitative).

Étapes :
1) Tri stable
Nous trions movies_text par movieId puis réinitialisons l’index.
Objectif : garantir un ordre déterministe (reproductible) entre les exécutions, ce qui stabilise les correspondances id ↔ index.

2) Extraction des listes
- movie_ids : liste ordonnée des movieId utilisés (une entrée par film).
- texts : liste ordonnée des textes associés (même ordre que movie_ids).
Objectif : alimenter directement les fonctions de calcul d’embeddings (un texte → un vecteur).

3) Dictionnaires de mapping
- movieid2idx : movieId → index (position dans la matrice d’embeddings).
- idx2movieid : index → movieId (inverse, utile pour décoder les recommandations).
- movieid2title : movieId → title (affichage lisible dans inspect_user).

4) Vérification simple
Nous affichons le nombre de films et de textes pour vérifier qu’il y a bien un texte par film (alignement 1–1).


-----

## Fonctions embeddings et evaluation


In [21]:
def get_cache_path(model_key):
    if not USE_DISK_CACHE:
        return None
    os.makedirs(CACHE_DIR, exist_ok=True)
    return os.path.join(CACHE_DIR, f"{model_key}.npy")


def load_embeddings_cache(cache_path):
    if cache_path and os.path.exists(cache_path):
        emb = np.load(cache_path)
        print("Loaded embeddings:", cache_path, emb.shape)
        return emb
    return None


def save_embeddings_cache(cache_path, emb):
    if cache_path:
        np.save(cache_path, emb)


def compute_item_embeddings_st(model_path, texts, batch_size=BATCH_SIZE, device=DEVICE):
    # Compute normalized item embeddings with SentenceTransformers.
    print(f"=== Model: {model_path} ===")
    model = SentenceTransformer(model_path, device=device)
    emb = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    emb = emb.astype("float32", copy=False)
    print("Embeddings shape:", emb.shape)
    return emb


def compute_item_embeddings_qwen(
    model_name,
    texts,
    batch_size=QWEN_BATCH_SIZE,
    max_length=QWEN_MAX_LENGTH,
    device=DEVICE,
):
    # Compute normalized item embeddings using a Qwen embedding model.
    print(f"=== Qwen model: {model_name} ===")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
    model.to(device)
    model.eval()

    all_embeddings = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start : start + batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}
            outputs = model(**encoded)

            token_embeddings = outputs.last_hidden_state
            attention_mask = encoded["attention_mask"].unsqueeze(-1)
            masked = token_embeddings * attention_mask
            summed = masked.sum(dim=1)
            counts = attention_mask.sum(dim=1).clamp(min=1e-9)
            pooled = summed / counts
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            all_embeddings.append(pooled.cpu().numpy())

    emb = np.vstack(all_embeddings).astype("float32", copy=False)
    print("Embeddings shape:", emb.shape)
    return emb


def build_user_item_dict(ratings_df, min_rating_positive=None):
    # Map user -> set of movieIds; optionally keep only positives.
    if min_rating_positive is None:
        df = ratings_df
    else:
        df = ratings_df[ratings_df["rating"] >= min_rating_positive]
    return df.groupby("userId")["movieId"].apply(set).to_dict()


def build_user_interactions(ratings_df, min_rating_positive=None):
    # Map user -> list of (movieId, rating, timestamp).
    if min_rating_positive is None:
        df = ratings_df
    else:
        df = ratings_df[ratings_df["rating"] >= min_rating_positive]

    has_ts = "timestamp" in df.columns
    cols = ["userId", "movieId", "rating"] + (["timestamp"] if has_ts else [])
    df = df[cols].copy()
    if has_ts:
        df = df.sort_values(["userId", "timestamp"])

    if has_ts:
        grouped = (
            df.groupby("userId")[["movieId", "rating", "timestamp"]]
            .apply(lambda x: list(map(tuple, x.values)))
            .to_dict()
        )
    else:
        grouped = (
            df.groupby("userId")[["movieId", "rating"]]
            .apply(lambda x: [(int(a), float(b), None) for a, b in x.values])
            .to_dict()
        )
    return grouped


def build_user_embeddings(
    train_user_data,
    item_embeddings,
    movieid2idx,
    user_profile_mode=USER_PROFILE_MODE,
    recency_lambda=RECENCY_LAMBDA,
):
    # Build user embeddings with optional rating/recency weighting.
    if not hasattr(build_user_embeddings, "_missing_ts_warned"):
        build_user_embeddings._missing_ts_warned = False

    user_embeddings = {}
    for uid, items in train_user_data.items():
        if not items:
            continue

        is_plain_ids = isinstance(items, set)
        if isinstance(items, list) and items:
            is_plain_ids = is_plain_ids or isinstance(items[0], (int, np.integer))

        if is_plain_ids:
            movie_ids = list(items)
            idxs = [movieid2idx[mid] for mid in movie_ids if mid in movieid2idx]
            if not idxs:
                continue
            emb = item_embeddings[idxs].mean(axis=0)
        else:
            idxs = []
            ratings = []
            timestamps = []
            for entry in items:
                if len(entry) >= 3:
                    mid, rating, ts = entry[0], entry[1], entry[2]
                else:
                    mid, rating, ts = entry[0], entry[1], None

                if ts is not None and pd.isna(ts):
                    ts = None

                if mid not in movieid2idx:
                    continue
                idxs.append(movieid2idx[mid])
                ratings.append(float(rating))
                timestamps.append(ts)

            if not idxs:
                continue

            mode = user_profile_mode
            ts_vals = [t for t in timestamps if t is not None]
            t_max = max(ts_vals) if ts_vals else None
            if mode in ("recency", "rating_recency") and t_max is None:
                if not build_user_embeddings._missing_ts_warned:
                    print("Recency mode requested but timestamp missing -> fallback")
                    build_user_embeddings._missing_ts_warned = True
                mode = "rating" if mode == "rating_recency" else "mean"

            if mode == "mean":
                emb = item_embeddings[idxs].mean(axis=0)
            else:
                weights = []
                for rating, ts in zip(ratings, timestamps):
                    w = 1.0
                    if mode in ("rating", "rating_recency"):
                        w *= rating
                    if mode in ("recency", "rating_recency"):
                        age_seconds = float(t_max - ts) if ts is not None else 0.0
                        age = age_seconds / 86400.0
                        w *= float(np.exp(-recency_lambda * age))
                    weights.append(w)
                weights = np.asarray(weights, dtype="float32")
                if weights.sum() <= 0:
                    emb = item_embeddings[idxs].mean(axis=0)
                else:
                    emb = np.average(item_embeddings[idxs], axis=0, weights=weights)

        norm = np.linalg.norm(emb)
        if norm > 0:
            emb = emb / norm
        user_embeddings[uid] = emb

    return user_embeddings


def build_popularity_scores(train_ratings, movieid2idx):
    # Log-count popularity score aligned with item indices.
    counts = train_ratings["movieId"].value_counts()
    pop_score = np.zeros(len(movieid2idx), dtype="float32")
    for mid, count in counts.items():
        idx = movieid2idx.get(int(mid))
        if idx is not None:
            pop_score[idx] = np.log1p(count)

    min_v = float(pop_score.min())
    max_v = float(pop_score.max())
    if max_v > min_v:
        pop_score = (pop_score - min_v) / (max_v - min_v)
    return pop_score


def recommend_for_user(
    user_id,
    user_embedding,
    item_embeddings,
    train_user_items_all,
    movieid2idx,
    idx2movieid,
    top_k=10,
):
    # Recommend top_k items by cosine similarity, excluding seen items.
    scores = item_embeddings @ user_embedding

    if USE_HYBRID:
        if pop_score_by_idx is None:
            if not hasattr(recommend_for_user, "_hybrid_warned"):
                print("USE_HYBRID=True but pop_score_by_idx is None -> content-only")
                recommend_for_user._hybrid_warned = True
        elif len(pop_score_by_idx) == len(scores):
            scores = ALPHA_HYBRID * scores + (1.0 - ALPHA_HYBRID) * pop_score_by_idx
        else:
            if not hasattr(recommend_for_user, "_hybrid_len_warned"):
                print("pop_score_by_idx length mismatch -> content-only")
                recommend_for_user._hybrid_len_warned = True

    seen = train_user_items_all.get(user_id, set())
    if seen:
        seen_idx = [movieid2idx[mid] for mid in seen if mid in movieid2idx]
        scores[seen_idx] = -np.inf

    if top_k >= len(scores):
        top_idx = np.argsort(scores)[::-1]
    else:
        top_idx = np.argpartition(scores, -top_k)[-top_k:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

    return [idx2movieid[idx] for idx in top_idx]


def recall_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0
    rec_k = recommended[:k]
    return len(set(rec_k) & relevant) / len(relevant)


def ndcg_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    dcg = 0.0
    for i, item in enumerate(rec_k):
        if item in relevant:
            dcg += 1.0 / np.log2(i + 2)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0


def map_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0
    rec_k = recommended[:k]
    hits = 0
    ap = 0.0
    for i, item in enumerate(rec_k, start=1):
        if item in relevant:
            hits += 1
            ap += hits / i
    return ap / min(len(relevant), k)


def hit_rate_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    return 1.0 if len(set(rec_k) & relevant) > 0 else 0.0


def get_eval_users(
    train_user_data,
    test_user_items,
    max_users=EVAL_MAX_USERS,
    random_state=RANDOM_STATE,
):
    users = [u for u in train_user_data if u in test_user_items and len(test_user_items[u]) > 0]
    if max_users is not None and len(users) > max_users:
        rng = np.random.default_rng(random_state)
        users = rng.choice(users, size=max_users, replace=False)
    return list(users)


def evaluate_model(
    item_embeddings,
    train_user_interactions,
    train_user_items_all,
    test_user_items,
    movieid2idx,
    idx2movieid,
    k_list,
    max_users=EVAL_MAX_USERS,
    random_state=RANDOM_STATE,
):
    global LAST_EVAL_USERS

    user_embeddings = build_user_embeddings(
        train_user_interactions, item_embeddings, movieid2idx
    )

    users = get_eval_users(
        train_user_interactions, test_user_items, max_users=max_users, random_state=random_state
    )
    users = [u for u in users if u in user_embeddings]

    LAST_EVAL_USERS = len(users)
    print("Users eval:", len(users))

    metrics = {f"Recall@{k}": [] for k in k_list}
    metrics.update({f"NDCG@{k}": [] for k in k_list})
    metrics.update({f"MAP@{k}": [] for k in k_list})
    metrics.update({f"HR@{k}": [] for k in k_list})

    max_k = max(k_list)
    for uid in users:
        recs = recommend_for_user(
            uid,
            user_embeddings[uid],
            item_embeddings,
            train_user_items_all,
            movieid2idx,
            idx2movieid,
            top_k=max_k,
        )
        relevant = test_user_items[uid]

        for k in k_list:
            metrics[f"Recall@{k}"].append(recall_at_k(recs, relevant, k))
            metrics[f"NDCG@{k}"].append(ndcg_at_k(recs, relevant, k))
            metrics[f"MAP@{k}"].append(map_at_k(recs, relevant, k))
            metrics[f"HR@{k}"].append(hit_rate_at_k(recs, relevant, k))

    return {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()}


def build_popularity_ranking(train_ratings):
    return train_ratings["movieId"].value_counts().index.tolist()


def evaluate_popularity_baseline(
    popular_items,
    train_user_items_all,
    test_user_items,
    k_list,
    max_users=EVAL_MAX_USERS,
    random_state=RANDOM_STATE,
):
    users = [u for u in test_user_items if u in train_user_items_all and len(test_user_items[u]) > 0]
    if max_users is not None and len(users) > max_users:
        rng = np.random.default_rng(random_state)
        users = rng.choice(users, size=max_users, replace=False)

    print("Users eval (popularity):", len(users))

    metrics = {f"Recall@{k}": [] for k in k_list}
    metrics.update({f"NDCG@{k}": [] for k in k_list})
    metrics.update({f"MAP@{k}": [] for k in k_list})
    metrics.update({f"HR@{k}": [] for k in k_list})

    max_k = max(k_list)
    for uid in users:
        seen = train_user_items_all.get(uid, set())
        recs = []
        for mid in popular_items:
            if mid in seen:
                continue
            recs.append(mid)
            if len(recs) >= max_k:
                break

        relevant = test_user_items[uid]
        for k in k_list:
            metrics[f"Recall@{k}"].append(recall_at_k(recs, relevant, k))
            metrics[f"NDCG@{k}"].append(ndcg_at_k(recs, relevant, k))
            metrics[f"MAP@{k}"].append(map_at_k(recs, relevant, k))
            metrics[f"HR@{k}"].append(hit_rate_at_k(recs, relevant, k))

    return {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()}


def score_all_items_cornac(model, user_idx):
    if hasattr(model, "score"):
        try:
            scores = model.score(user_idx)
            scores = np.asarray(scores)
            if scores.ndim == 1:
                return scores
        except Exception:
            pass

    if hasattr(model, "U") and hasattr(model, "V"):
        scores = model.U[user_idx] @ model.V.T
        if hasattr(model, "bu"):
            scores = scores + model.bu[user_idx]
        if hasattr(model, "bi"):
            scores = scores + model.bi
        return np.asarray(scores)

    raise RuntimeError("Cornac model does not support score-all-items")


def evaluate_cornac_model(
    model,
    train_set,
    train_user_items_all,
    test_user_items,
    k_list,
    max_users=EVAL_MAX_USERS,
    random_state=RANDOM_STATE,
):
    uid_map = train_set.uid_map
    iid_map = train_set.iid_map
    idx2item = {idx: item for item, idx in iid_map.items()}

    users = [u for u in test_user_items if u in uid_map and len(test_user_items[u]) > 0]
    if max_users is not None and len(users) > max_users:
        rng = np.random.default_rng(random_state)
        users = rng.choice(users, size=max_users, replace=False)

    print("Users eval (cornac):", len(users))

    metrics = {f"Recall@{k}": [] for k in k_list}
    metrics.update({f"NDCG@{k}": [] for k in k_list})
    metrics.update({f"MAP@{k}": [] for k in k_list})
    metrics.update({f"HR@{k}": [] for k in k_list})

    max_k = max(k_list)
    for uid in users:
        uidx = uid_map[uid]
        scores = score_all_items_cornac(model, uidx)
        scores = np.array(scores, dtype="float32", copy=True)

        seen = train_user_items_all.get(uid, set())
        if seen:
            seen_idx = [iid_map[mid] for mid in seen if mid in iid_map]
            scores[seen_idx] = -np.inf

        if max_k >= len(scores):
            top_idx = np.argsort(scores)[::-1]
        else:
            top_idx = np.argpartition(scores, -max_k)[-max_k:]
            top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

        recs = [idx2item[idx] for idx in top_idx]
        relevant = test_user_items[uid]

        for k in k_list:
            metrics[f"Recall@{k}"].append(recall_at_k(recs, relevant, k))
            metrics[f"NDCG@{k}"].append(ndcg_at_k(recs, relevant, k))
            metrics[f"MAP@{k}"].append(map_at_k(recs, relevant, k))
            metrics[f"HR@{k}"].append(hit_rate_at_k(recs, relevant, k))

    return {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()}


def inspect_user(user_id, model_key, top_n=10):
    # Quick qualitative inspection: liked items vs top-N recommendations.
    if model_key in ("qwen", "qwen-0.6B"):
        model_key = "qwen-0.6B"

    if user_id not in train_user_items_all:
        print("Unknown user_id:", user_id)
        return

    if model_key not in item_embeddings_cache:
        if model_key in EMBEDDING_MODELS:
            print("Computing embeddings for:", model_key)
            cache_path = get_cache_path(model_key)
            emb = load_embeddings_cache(cache_path)
            if emb is None:
                emb = compute_item_embeddings_st(
                    EMBEDDING_MODELS[model_key],
                    texts,
                    batch_size=BATCH_SIZE,
                    device=DEVICE,
                )
                save_embeddings_cache(cache_path, emb)
            item_embeddings_cache[model_key] = emb
        elif model_key == "qwen-0.6B":
            if not RUN_QWEN:
                print("RUN_QWEN=False; set it True and run Qwen embeddings first.")
                return
            cache_path = get_cache_path(model_key)
            emb = load_embeddings_cache(cache_path)
            if emb is None:
                emb = compute_item_embeddings_qwen(
                    QWEN_MODEL_NAME,
                    texts,
                    batch_size=QWEN_BATCH_SIZE,
                    max_length=QWEN_MAX_LENGTH,
                    device=DEVICE,
                )
                save_embeddings_cache(cache_path, emb)
            item_embeddings_cache[model_key] = emb
        else:
            print("Unknown model_key:", model_key)
            return

    interactions = train_user_interactions.get(user_id, [])
    if not interactions:
        print("No interactions for user:", user_id)
        return

    user_emb = build_user_embeddings(
        {user_id: interactions},
        item_embeddings_cache[model_key],
        movieid2idx,
    ).get(user_id)
    if user_emb is None:
        print("Cannot build user embedding for:", user_id)
        return

    recs = recommend_for_user(
        user_id,
        user_emb,
        item_embeddings_cache[model_key],
        train_user_items_all,
        movieid2idx,
        idx2movieid,
        top_k=top_n,
    )

    def _format_titles(ids, limit=10):
        titles = [movieid2title.get(mid, str(mid)) for mid in ids]
        if len(titles) > limit:
            return titles[:limit] + ["..."]
        return titles

    liked = train_user_items_pos.get(user_id, set())
    print("User:", user_id)
    print("Train likes:", _format_titles(sorted(liked), limit=top_n))
    test_likes = test_user_items.get(user_id, set())
    if test_likes:
        print("Test likes :", _format_titles(sorted(test_likes), limit=top_n))
    print("Top-N recs :", _format_titles(recs, limit=top_n))


Ce bloc regroupe le cœur “moteur” du système : (1) calcul et mise en cache des embeddings de films, (2) construction des profils utilisateurs, (3) génération des recommandations, et (4) évaluation (Recall/NDCG/MAP/HR) + baselines (popularité, Cornac).

1) Cache disque des embeddings (accélération / reproductibilité)
- get_cache_path() : définit un chemin de cache par modèle (ex. embeddings_cache/mpnet.npy) et crée le dossier si besoin.
- load_embeddings_cache() / save_embeddings_cache() : recharge ou sauvegarde les matrices d’embeddings. Objectif : éviter de recalculer des vecteurs coûteux à chaque exécution et garantir des runs comparables.

2) Calcul des embeddings d’items (films)
- compute_item_embeddings_st() : calcule des embeddings normalisés via SentenceTransformers (MiniLM/MPNet/bge-small…). Normalisation L2 activée pour que la similarité cosinus se résume à un produit scalaire.
- compute_item_embeddings_qwen() : alternative GPU plus lourde basée sur Qwen (tokenisation + mean pooling masqué + normalisation). À activer uniquement pour un run dédié (RUN_QWEN=True).

3) Structures utilisateur → items / interactions
- build_user_item_dict() : construit userId → set(movieId). Optionnellement, ne conserve que les “positifs” (rating ≥ seuil).
- build_user_interactions() : construit userId → liste ordonnée (movieId, rating, timestamp). Utile pour des profils pondérés (rating / récence) et pour un split temporel cohérent.

4) Construction des profils utilisateurs (user embeddings)
- build_user_embeddings() : calcule un vecteur par utilisateur à partir des embeddings des films vus en train.
  Modes disponibles :
  - mean : moyenne simple
  - rating : pondération par la note
  - recency : pondération exponentielle selon l’ancienneté (plus récent = plus lourd)
  - rating_recency : combinaison des deux
  Le profil est ensuite normalisé (L2) pour rester compatible avec une similarité cosinus stable.
  En l’absence de timestamps, les modes “recency” basculent automatiquement sur une option sûre (fallback contrôlé).

5) Baseline popularité et option hybride
- build_popularity_scores() : calcule un score de popularité (log(1+count)), puis le normalise entre 0 et 1.
- recommend_for_user() : score chaque film par similarité (item_embeddings @ user_embedding), exclut les films déjà vus, puis renvoie le Top-K.
  Si USE_HYBRID=True : mélange content-based et popularité :
  score_final = ALPHA_HYBRID * score_content + (1-ALPHA_HYBRID) * score_pop
  Des garde-fous évitent les crashs si pop_score_by_idx est absent ou mal aligné.

6) Métriques Top-K (évaluation)
- recall_at_k : couverture des items pertinents retrouvés
- ndcg_at_k : qualité du ranking (hits en haut = mieux)
- map_at_k : précision moyenne sur le Top-K (favorise la justesse aux premiers rangs)
- hit_rate_at_k : au moins un hit dans le Top-K (métrique binaire)

7) Sélection des utilisateurs et évaluation globale
- get_eval_users() : sélectionne les utilisateurs évaluables (présents en train + test, avec au moins 1 item pertinent). Option : sous-échantillonnage (EVAL_MAX_USERS) pour accélérer.
- evaluate_model() : calcule les profils utilisateurs, produit des recommandations, puis agrège les métriques (moyenne) sur l’ensemble des utilisateurs évalués.
- build_popularity_ranking() / evaluate_popularity_baseline() : baseline “top populaires” avec exclusion des items déjà vus.
- evaluate_cornac_model() : évaluation MF/BPR Cornac en scorant tous les items, en masquant les items vus, puis en calculant les mêmes métriques Top-K (comparaison homogène).

8) Inspection qualitative (sanity check)
- inspect_user() : affiche, pour un utilisateur donné, ses “likes” train/test et les Top-N recommandations d’un modèle.
Objectif : vérifier qualitativement que le système produit des recommandations cohérentes, et diagnostiquer rapidement des erreurs (mauvais mapping, fuite, exclusion des vus, etc.).


-----

## Calcul des embeddings et evaluation


In [22]:
# Sanity checks / run order
required = [
    "ratings_small",
    "movies_text",
    "texts",
    "movieid2idx",
    "idx2movieid",
    "train_ratings",
    "test_ratings",
    "SPLIT_STRATEGY_USED",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Run order issue: missing {missing}. Execute the cells above in order."
    )

if len(texts) == 0 or len(movieid2idx) == 0:
    raise RuntimeError("Empty texts or mappings; check filtering thresholds.")

print("Sanity checks OK: data and mappings ready.")


Sanity checks OK: data and mappings ready.


Ce bloc sécurise l’exécution du notebook avant de lancer l’entraînement/évaluation.

- Nous vérifions que les objets indispensables ont bien été créés par les cellules précédentes :
  ratings_small, movies_text, texts, movieid2idx, idx2movieid, train_ratings, test_ratings, SPLIT_STRATEGY_USED.
  Si un élément manque, nous arrêtons immédiatement avec un message clair indiquant quelle cellule doit être rejouée.

- Nous ajoutons un contrôle de cohérence minimal :
  si la liste des textes (items) ou les mappings movieId ↔ index sont vides, cela indique généralement un filtrage trop agressif
  (seuils MIN_USER_RATINGS / MIN_ITEM_RATINGS) ou un problème dans la construction de movies_text.

- Si tout est correct, nous confirmons explicitement que les données et les mappings sont prêts pour la suite :
  “Sanity checks OK: data and mappings ready.”


------

In [23]:
train_user_items_all = build_user_item_dict(train_ratings, min_rating_positive=None)
train_user_items_pos = build_user_item_dict(train_ratings, min_rating_positive=MIN_RATING_POSITIVE)
train_user_interactions = build_user_interactions(
    train_ratings, min_rating_positive=MIN_RATING_POSITIVE
)
test_user_items = build_user_item_dict(test_ratings, min_rating_positive=MIN_RATING_POSITIVE)

print("Users with train interactions :", len(train_user_items_all))
print("Users with train positives    :", len(train_user_items_pos))
print("Users with test positives     :", len(test_user_items))

if USE_HYBRID:
    pop_score_by_idx = build_popularity_scores(train_ratings, movieid2idx)
    print("Hybrid enabled, alpha:", ALPHA_HYBRID)
else:
    pop_score_by_idx = None

item_embeddings_cache = {}
all_results = {}

for model_key in ACTIVE_MODELS:
    if model_key not in EMBEDDING_MODELS:
        raise ValueError(f"Unknown model key: {model_key}")

    model_path = EMBEDDING_MODELS[model_key]
    cache_path = get_cache_path(model_key)
    emb = load_embeddings_cache(cache_path)

    if emb is None:
        emb = compute_item_embeddings_st(
            model_path,
            texts,
            batch_size=BATCH_SIZE,
            device=DEVICE,
        )
        save_embeddings_cache(cache_path, emb)

    item_embeddings_cache[model_key] = emb

    metrics = evaluate_model(
        item_embeddings_cache[model_key],
        train_user_interactions,
        train_user_items_all,
        test_user_items,
        movieid2idx,
        idx2movieid,
        k_list=K_LIST,
        max_users=EVAL_MAX_USERS,
        random_state=RANDOM_STATE,
    )
    all_results[model_key] = metrics

    if USE_HYBRID and RUN_ALPHA_GRID:
        alpha_backup = ALPHA_HYBRID
        for alpha in ALPHA_GRID:
            ALPHA_HYBRID = float(alpha)
            metrics = evaluate_model(
                item_embeddings_cache[model_key],
                train_user_interactions,
                train_user_items_all,
                test_user_items,
                movieid2idx,
                idx2movieid,
                k_list=K_LIST,
                max_users=EVAL_MAX_USERS,
                random_state=RANDOM_STATE,
            )
            all_results[f"{model_key}_hybrid_a{alpha}"] = metrics
        ALPHA_HYBRID = alpha_backup

if RUN_QWEN:
    qwen_key = "qwen-0.6B"
    cache_path = get_cache_path(qwen_key)
    emb = load_embeddings_cache(cache_path)

    if emb is None:
        emb = compute_item_embeddings_qwen(
            QWEN_MODEL_NAME,
            texts,
            batch_size=QWEN_BATCH_SIZE,
            max_length=QWEN_MAX_LENGTH,
            device=DEVICE,
        )
        save_embeddings_cache(cache_path, emb)

    item_embeddings_cache[qwen_key] = emb

    metrics = evaluate_model(
        item_embeddings_cache[qwen_key],
        train_user_interactions,
        train_user_items_all,
        test_user_items,
        movieid2idx,
        idx2movieid,
        k_list=K_LIST,
        max_users=EVAL_MAX_USERS,
        random_state=RANDOM_STATE,
    )
    all_results[qwen_key] = metrics

if RUN_POPULARITY_BASELINE:
    pop_ranking = build_popularity_ranking(train_ratings)
    metrics = evaluate_popularity_baseline(
        pop_ranking,
        train_user_items_all,
        test_user_items,
        k_list=K_LIST,
        max_users=EVAL_MAX_USERS,
        random_state=RANDOM_STATE,
    )
    all_results["popularity"] = metrics

if RUN_CORNAC_BASELINES:
    try:
        from cornac.data import Dataset
        from cornac.models import MF, BPR

        train_uir = list(
            train_ratings[["userId", "movieId", "rating"]].itertuples(index=False, name=None)
        )
        train_set = Dataset.from_uir(train_uir)

        if "mf" in CORNAC_MODELS:
            mf = MF(
                k=CORNAC_FACTORS,
                max_iter=CORNAC_MAX_ITER,
                learning_rate=CORNAC_LR,
                lambda_reg=CORNAC_REG,
                use_bias=True,
                seed=RANDOM_STATE,
                verbose=True,
            )
            mf.fit(train_set)
            metrics = evaluate_cornac_model(
                mf,
                train_set,
                train_user_items_all,
                test_user_items,
                k_list=K_LIST,
                max_users=CORNAC_MAX_USERS,
                random_state=RANDOM_STATE,
            )
            all_results["cornac_mf"] = metrics

        if "bpr" in CORNAC_MODELS:
            train_uir_pos = list(
                train_ratings[["userId", "movieId"]]
                .assign(rating=1.0)
                .itertuples(index=False, name=None)
            )
            train_set_pos = Dataset.from_uir(train_uir_pos)
            bpr = BPR(
                k=CORNAC_FACTORS,
                max_iter=CORNAC_MAX_ITER,
                learning_rate=CORNAC_LR,
                lambda_reg=CORNAC_REG,
                seed=RANDOM_STATE,
                verbose=True,
            )
            bpr.fit(train_set_pos)
            metrics = evaluate_cornac_model(
                bpr,
                train_set_pos,
                train_user_items_all,
                test_user_items,
                k_list=K_LIST,
                max_users=CORNAC_MAX_USERS,
                random_state=RANDOM_STATE,
            )
            all_results["cornac_bpr"] = metrics
    except Exception as exc:
        print("Cornac baseline failed:", repr(exc))

results_df = pd.DataFrame(all_results).T

RUN_CONFIG = {
    "run_time": datetime.utcnow().isoformat() + "Z",
    "random_state": RANDOM_STATE,
    "min_user_ratings": int(MIN_USER_RATINGS),
    "min_item_ratings": int(MIN_ITEM_RATINGS),
    "max_interactions": int(MAX_INTERACTIONS) if MAX_INTERACTIONS is not None else None,
    "test_size": float(TEST_SIZE),
    "split_strategy": SPLIT_STRATEGY,
    "split_strategy_used": SPLIT_STRATEGY_USED,
    "has_timestamp": bool("timestamp" in ratings_small.columns),
    "active_models": list(ACTIVE_MODELS),
    "user_profile_mode": USER_PROFILE_MODE,
    "recency_lambda": float(RECENCY_LAMBDA),
    "use_hybrid": bool(USE_HYBRID),
    "alpha_hybrid": float(ALPHA_HYBRID),
    "alpha_grid": list(ALPHA_GRID),
    "k_list": list(K_LIST),
    "eval_max_users": EVAL_MAX_USERS,
    "n_users": int(ratings_small["userId"].nunique()),
    "n_items": int(ratings_small["movieId"].nunique()),
    "n_train": int(len(train_ratings)),
    "n_test": int(len(test_ratings)),
    "n_train_users": int(train_ratings["userId"].nunique()),
    "n_test_users": int(test_ratings["userId"].nunique()),
    "n_eval_users": int(LAST_EVAL_USERS),
    "device": DEVICE,
}

results_df

Users with train interactions : 138493
Users with train positives    : 138134
Users with test positives     : 135583
Hybrid enabled, alpha: 0.7
=== Model: sentence-transformers/all-MiniLM-L6-v2 ===


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Optimization finished!
Users eval (cornac): 135583


  0%|          | 0/50 [00:00<?, ?it/s]

Optimization finished!
Users eval (cornac): 135583


/tmp/ipykernel_55/1091025314.py:177: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_time": datetime.utcnow().isoformat() + "Z",


,Recall@5,Recall@10,Recall@20,NDCG@5,NDCG@10,NDCG@20,MAP@5,MAP@10,MAP@20,HR@5,HR@10,HR@20
miniLM,0.033670,0.056209,0.090480,0.070726,0.070573,0.078075,0.039769,0.031523,0.029599,0.249295,0.361478,0.491723
mpnet,0.036395,0.061158,0.099254,0.074664,0.075960,0.085339,0.042401,0.034414,0.032828,0.259241,0.378077,0.513764
bge-small,0.034572,0.060134,0.101374,0.072443,0.075202,0.085977,0.041187,0.034328,0.033226,0.251133,0.366455,0.501122
popularity,0.036237,0.060291,0.096463,0.074733,0.075878,0.084164,0.044423,0.036913,0.035239,0.241889,0.338899,0.449216
cornac_mf,0.011489,0.020188,0.033750,0.023353,0.025209,0.029312,0.012433,0.010495,0.010127,0.092475,0.152172,0.232787
cornac_bpr,0.041968,0.073482,0.121374,0.090635,0.093743,0.105714,0.054392,0.045613,0.044045,0.287079,0.407345,0.533806


Ce bloc exécute l’ensemble du pipeline d’évaluation, de manière reproductible, sur le split train/test construit plus haut.

1) Préparation des ensembles utilisateur/items
- Nous construisons :
  • train_user_items_all : tous les films vus en train (pour exclure les items déjà consommés).
  • train_user_items_pos : les films “positifs” en train (rating ≥ MIN_RATING_POSITIVE) pour définir les préférences.
  • train_user_interactions : liste ordonnée des interactions positives (movieId, rating, timestamp) utilisée pour construire les profils utilisateurs.
  • test_user_items : les films “positifs” en test, utilisés comme vérité terrain pour l’évaluation Top-K.
Ces structures accélèrent le scoring et garantissent une évaluation cohérente (pas de recommandation d’items déjà vus).

2) Option hybride (contenu + popularité)
Si USE_HYBRID=True, nous calculons pop_score_by_idx (popularité normalisée alignée sur l’indexation des items) puis nous combinons :
score_final = ALPHA_HYBRID * score_contenu + (1 - ALPHA_HYBRID) * score_popularité.
Objectif : réduire le bruit du contenu textuel et stabiliser les recommandations.

3) Embeddings : calcul + cache disque
Pour chaque modèle de ACTIVE_MODELS :
- Nous essayons de charger les embeddings depuis CACHE_DIR (gain de temps lors des reruns).
- Sinon, nous calculons les embeddings sur texts via SentenceTransformers, normalisés L2, puis nous les sauvegardons.
Cela rend le notebook plus rapide et plus “prod-ready” (ré-exécutable sans recalcul lourd systématique).

4) Évaluation Top-K des modèles embeddings
Nous évaluons chaque modèle avec evaluate_model :
- Construction d’un embedding utilisateur à partir des interactions positives du train (profil utilisateur).
- Recommandation Top-K par similarité cosinus, en excluant les items déjà vus.
- Calcul et agrégation des métriques : Recall@K, NDCG@K, MAP@K, HitRate@K.
Le nombre d’utilisateurs évalués est stocké dans LAST_EVAL_USERS (audit du périmètre réel d’évaluation).

5) Baseline Popularité
Si RUN_POPULARITY_BASELINE=True :
- Nous construisons un ranking global des items les plus vus dans le train.
- Nous évaluons ce ranking en excluant les items déjà vus par utilisateur.
Cette baseline sert de point de repère : elle est souvent très forte sur MovieLens.

6) Baselines collaboratives (Cornac)
Si RUN_CORNAC_BASELINES=True :
- MF est entraîné sur (userId, movieId, rating).
- BPR est entraîné en implicite sur les interactions (userId, movieId) positives (rating=1).
Nous évaluons ensuite en scorant tous les items pour chaque utilisateur (score-all-items), avec exclusion des items vus en train.

7) Consolidation et traçabilité
- Nous rassemblons toutes les métriques dans results_df (une ligne par modèle).
- Nous construisons RUN_CONFIG : paramètres clés (seed, split utilisé, seuils, modèles évalués, tailles train/test, device, etc.).
Ces artefacts permettent d’exporter un résultat auditable (CSV + JSON) associé à une configuration unique.

Lecture rapide des résultats (extrait)
- Cornac BPR est le meilleur : Recall@10 ≈ 0.0735 ; NDCG@10 ≈ 0.0937.
- Les embeddings (MPNet / bge-small) sont proches de la popularité sur Recall@10 et NDCG@10.
- MF est nettement en dessous, ce qui est attendu sur un setup ranking Top-K comparé à BPR.


-----

## Synthese des resultats


In [24]:
print(results_df.round(4))
if "Recall@10" in results_df.columns and "NDCG@10" in results_df.columns:
    summary_df = results_df.sort_values(
        by=["Recall@10", "NDCG@10"], ascending=False
    )
    print(summary_df[["Recall@10", "NDCG@10"]].round(4))

# Optional: save summary for report/README
results_df.to_csv("results_summary.csv", index=True)

# Export run configuration
if RUN_CONFIG:
    with open("run_config.json", "w", encoding="utf-8") as f:
        json.dump(RUN_CONFIG, f, indent=2)


            Recall@5  Recall@10  Recall@20  NDCG@5  NDCG@10  NDCG@20   MAP@5  \
miniLM        0.0337     0.0562     0.0905  0.0707   0.0706   0.0781  0.0398   
mpnet         0.0364     0.0612     0.0993  0.0747   0.0760   0.0853  0.0424   
bge-small     0.0346     0.0601     0.1014  0.0724   0.0752   0.0860  0.0412   
popularity    0.0362     0.0603     0.0965  0.0747   0.0759   0.0842  0.0444   
cornac_mf     0.0115     0.0202     0.0338  0.0234   0.0252   0.0293  0.0124   
cornac_bpr    0.0420     0.0735     0.1214  0.0906   0.0937   0.1057  0.0544   

            MAP@10  MAP@20    HR@5   HR@10   HR@20  
miniLM      0.0315  0.0296  0.2493  0.3615  0.4917  
mpnet       0.0344  0.0328  0.2592  0.3781  0.5138  
bge-small   0.0343  0.0332  0.2511  0.3665  0.5011  
popularity  0.0369  0.0352  0.2419  0.3389  0.4492  
cornac_mf   0.0105  0.0101  0.0925  0.1522  0.2328  
cornac_bpr  0.0456  0.0440  0.2871  0.4073  0.5338  
            Recall@10  NDCG@10
cornac_bpr     0.0735   0.0937
mpnet 

Ce bloc finalise l’évaluation en produisant un rendu lisible, un classement des modèles, puis des exports prêts pour le rapport et GitHub.

1) Affichage des métriques
- Nous affichons results_df arrondi à 4 décimales afin de garder une sortie compacte et comparable (Recall/NDCG/MAP/HR @ {5,10,20}).

2) Classement “référence” sur Top-10
- Si Recall@10 et NDCG@10 sont disponibles, nous trions les modèles par performance décroissante :
  • critère principal : Recall@10
  • critère secondaire : NDCG@10
- Nous affichons ensuite uniquement (Recall@10, NDCG@10) pour obtenir un tableau de synthèse directement exploitable dans le rapport/README.

3) Export CSV (résultats consolidés)
- Nous exportons results_df en results_summary.csv (index inclus : une ligne = un modèle).
Ce fichier sert de source unique de vérité pour le rapport et évite les copier-coller manuels.

4) Export JSON (traçabilité du run)
- Si RUN_CONFIG est renseigné, nous l’exportons en run_config.json.
Ce fichier capture la configuration exacte du run (seed, split utilisé, seuils de filtrage, modèles actifs, tailles train/test, device, etc.) afin de rendre l’expérience auditables et reproductible.


-----

## Demo inspection utilisateur


In [25]:
if ACTIVE_MODELS and train_user_items_pos:
    sample_user = next(iter(train_user_items_pos))
    inspect_user(sample_user, ACTIVE_MODELS[0], top_n=10)

User: 1
Train likes: ['Rob Roy (1995)', 'Clerks (1994)', 'Interview with the Vampire: The Vampire Chronicles (1994)', 'Star Wars: Episode IV - A New Hope (1977)', 'Léon: The Professional (a.k.a. The Professional) (Léon) (1994)', 'Pulp Fiction (1994)', 'Shawshank Redemption, The (1994)', 'Blade Runner (1982)', 'Die Hard (1988)', 'Fish Called Wanda, A (1988)', '...']
Test likes : ['Dune (1984)', 'Splash (1984)', 'Legend (1985)', 'Star Wars: Episode I - The Phantom Menace (1999)', 'Hook (1991)', 'Highlander: Endgame (Highlander IV) (2000)', 'Jabberwocky (1977)', 'Time Machine, The (2002)', 'Clash of the Titans (1981)', 'Underworld (2003)', '...']
Top-N recs : ['Back to the Future (1985)', 'Monsters, Inc. (2001)', 'Old Boy (2003)', 'Fifth Element, The (1997)', 'Conan the Barbarian (1982)', 'Heavy Metal (1981)', 'Matrix Reloaded, The (2003)', 'Jurassic Park (1993)', 'Reign of Fire (2002)', 'Fly, The (1986)']


Ce bloc réalise un contrôle “à la main” sur un utilisateur afin de compléter l’évaluation chiffrée par une vérification qualitative.

- Nous sélectionnons un utilisateur de train ayant au moins un film positif (train_user_items_pos) et lançons inspect_user() sur le premier modèle actif (ACTIVE_MODELS[0]).
- La fonction affiche trois éléments :
  1) Train likes : films aimés en entraînement (profil utilisateur appris à partir de ces items).
  2) Test likes  : films aimés en test (référence hors entraînement, utilisée pour juger la pertinence).
  3) Top-N recs  : recommandations produites par le modèle (top_n=10), en excluant les films déjà vus en entraînement.

Intérêt :
- Vérifier rapidement que les recommandations “font sens” au regard du profil (cohérence thématique/genre) et qu’elles ne recrachent pas des films déjà consommés.
- Donner un exemple concret à insérer dans le rapport/README pour illustrer le fonctionnement du système au-delà des métriques globales.


## Conclusion

Sur MovieLens 20M, les modèles collaboratifs restent les plus performants : Cornac BPR obtient les meilleurs scores en top-K (notamment Recall@10 = 0.0735 et NDCG@10 = 0.0937). Notre approche content-based par embeddings est néanmoins solide et compétitive : MPNet et bge-small atteignent des performances proches de la baseline popularité, tout en offrant une personnalisation plus explicable via la similarité sémantique entre profils utilisateurs et contenus des films. Enfin, l’inspection qualitative confirme la cohérence des recommandations produites, ce qui valide la logique du pipeline ; la suite la plus rentable consiste à hybrider contenu et signal collaboratif pour combiner interprétabilité et performance.